# Stage2_Hazel · Jev 原始 / 新校准 KD · 18条件

两种教师软标签 × α={0.25,0.5,0.75} × 三种子={20261010,20261011,20261012}。固定公共学习率 **0.0003**、蒸馏温度 **τ=2**。仅做开发集实验，不评估测试集。

本 notebook 自包含训练代码与本次新教师参数。上传到 Colab GPU，按顺序运行即可；无需另传 .py、原图或运行 Teacher。**这是未执行的正式训练 notebook，不包含正式新成绩。**

本次实际附件为 `stage2_D2交付.zip`，内部包含 `teacher_calibration_stage2.json`。内嵌的是附件原始文件字节，Jev `selected_params`：b=0.1499999999999999，T=1.3425725027802635。启动时校验文件哈希、教师版本、训练缓存与拟合集来源；不读取历史 b/T 用于新校准，不使用学生后校准温度，不重新校准。

输出完整保存18条件状态、最佳模型、每轮历史、运行协议、逐条预测、与Day2合并的指标表，以及三类配对比较：KD−调好学习率的CE、同α校准KD−原始KD、不同α差异。旧lr=.001 KD单列补充，不计入18条件。


## 1．挂载 Drive 与输入路径

- `SOURCE`：10月13日完整成果包的解压目录，含 reused_v1 和 文件校验清单.json，提供已有训练/开发特征及原始教师 logits。
- `DAY2`：Hazel 已完成的 **CE三学习率、两种LS补充实验**输出目录，或其导出 ZIP；需要 common_lr.json、supervised_selection.json、report、code 和15个best模型。不是本次教师校准交付包。
- `WORK`：新建独立目录。必须与所有输入相互独立，不能复用旧六模型 notebook 的 WORK。
- `LEGACY`：可选，原 lr=.001 CE/LS/KD 结果目录或 ZIP。默认在常用位置存在时自动接入旧验证报告；找不到则记录未提供，不影响18条件，旧文件保持原样。可手动修改。

代码会检查 Day2 的公共学习率确实为 .0003、输入缓存一致，并验证其模型/训练历史/报告/代码快照。支持 Day2 导出的轻量ZIP，不需要 val_fit 图像特征。


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
SOURCE=Path('/content/drive/MyDrive/1013实验成果包与结论/10月13日_任务交付')
DAY2=Path('/content/drive/MyDrive/CLIP_Jev_CE三学习率_两种LS_补充')
WORK=Path('/content/drive/MyDrive/Stage2_Hazel_KD18_20261024')
legacy_candidate=Path('/content/drive/MyDrive/CLIP_Jev_CE_LS_学生后校准_20261023')
LEGACY=legacy_candidate if legacy_candidate.exists() else None
for p in [SOURCE,DAY2]+([LEGACY] if LEGACY else []):
    assert p.exists(), f'请检查输入：{p}'
    assert not WORK.resolve().is_relative_to(p.resolve()) and not p.resolve().is_relative_to(WORK.resolve()), '输入输出必须独立'
WORK.mkdir(parents=True,exist_ok=True)
MAX_NEW_GROUPS=6  # 0=只整理/导出；1~6=本次最多启动多少个尚未完成的三种子配置


## 2．检查环境

本格仅检查GPU可用并打印版本，第4节才进行Day2完整环境比对。允许已确认的 Python 3.13.15→3.13.16 补丁升级，协议同时保留Day2版本、实际版本及差异，不声称逐位等价。其余 PyTorch、numpy、CUDA、cuDNN、GPU、精度等记录仍须一致。FP32，禁用TF32。仅在缺少pandas时安装它，不自动升级训练依赖。


In [ ]:
import os,sys,subprocess,importlib.util
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
if importlib.util.find_spec('pandas') is None:
    subprocess.run([sys.executable,'-m','pip','install','pandas'],check=True)
import torch,numpy as np,pandas as pd
assert torch.cuda.is_available(), '请选择 GPU 运行时'
print('torch:',torch.__version__,'numpy:',np.__version__,'GPU:',torch.cuda.get_device_name(0))


## 3．写入内嵌代码与新教师交付快照

只载入本 notebook 自带的训练模块；不执行交付 ZIP 中的脚本。教师参数保持附件原始字节并校验 SHA256。输入只读，代码和结果写入新WORK。


In [ ]:
import hashlib,importlib.util,base64
SOURCES={'cache_audit.py': '"""Read-only historical cache audit and disposable, unified training benchmark."""\nfrom pathlib import Path\nimport hashlib, json, time, math, random, platform, uuid, csv\nimport numpy as np\n\nREV = \'3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268\'\nMETHODS = [\'CE\', \'KD_raw\', \'KD_calibrated\', \'KD_base\', \'KD_hard\', \'CE_image\', \'CE_text\']\n\ndef check(ok, message):\n    if not ok: raise ValueError(message)\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(1048576), b\'\'): h.update(chunk)\n    return h.hexdigest()\n\ndef read(path): return json.loads(Path(path).read_text(encoding=\'utf-8-sig\'))\ndef rows(path): return [json.loads(s) for s in Path(path).read_text(encoding=\'utf-8-sig\').splitlines() if s.strip()]\ndef save(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.\' + uuid.uuid4().hex + \'.partial\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\ndef locate(source):\n    source = Path(source).resolve()\n    candidates = [source, source/\'reused_v1\']\n    candidates += list(source.glob(\'**/reused_v1\'))\n    matches = sorted(set(p for p in candidates if (p/\'students/training_lock.json\').is_file()))\n    check(len(matches) == 1, \'SOURCE 必须指向已解压的单个成果包或 Student_v1 目录；未唯一找到训练锁。\')\n    return matches[0]\n\ndef audit(source):\n    old = locate(source); lock = read(old/\'students/training_lock.json\')\n    files = {}; data = {}; seen = set(); seen_samples = set()\n    def record(p):\n        digest=sha(p); files[p.relative_to(old).as_posix()] = digest; return digest\n    revision = read(old/\'features/clip_revision.json\')\n    check(revision[\'revision\'] == REV, \'CLIP revision 不符\')\n    record(old/\'features/clip_revision.json\'); record(old/\'students/training_lock.json\')\n    # Only read held-out manifests to verify isolation; never read their features/predictions.\n    for split, n in [(\'train\',6000),(\'val_fit\',400),(\'val_select\',400),(\'test\',1000)]:\n        manifest = old/f\'handoff/manifests/{split}.jsonl\'; rr = rows(manifest); record(manifest)\n        ids = [str(r[\'image_id\']) for r in rr]; sid = [r[\'sample_id\'] for r in rr]\n        check(len(rr)==n and len(set(ids))==n and len(set(sid))==n, f\'{split} 数量/重复 ID 异常\')\n        check(not seen.intersection(ids) and not seen_samples.intersection(sid), \'划分间 ID 泄漏\')\n        seen.update(ids); seen_samples.update(sid)\n        check(all(r[\'label\'] in [0,1,2] and r[\'split\']==split for r in rr), \'标签或 split 异常\')\n        if split not in [\'train\',\'val_select\']: continue\n        p = old/f\'features/{split}.npz\'; meta_path = old/f\'features/{split}_meta.json\'\n        meta = read(meta_path); record(meta_path)\n        digest = record(p)\n        check(digest == meta[\'npz_sha256\'] == lock[\'feature_hashes\'][split], f\'{split} 特征哈希不符\')\n        check(meta[\'manifest_sha256\']==sha(manifest) and meta[\'revision\']==REV, \'特征来源不符\')\n        check(meta[\'feature_order\']==[\'v\',\'t\',\'abs(v-t)\',\'v*t\'] and meta[\'frozen_encoder\'], \'特征定义不符\')\n        with np.load(p, allow_pickle=False) as z: d = {k:z[k].copy() for k in [\'x\',\'y\',\'ids\',\'image_ids\']}\n        x = d[\'x\']; v,t = x[:,:512],x[:,512:1024]\n        check(x.shape==(n,2048) and x.dtype==np.float32 and np.isfinite(x).all(), \'特征形状/精度/有限性异常\')\n        check(d[\'ids\'].tolist()==sid and d[\'image_ids\'].astype(str).tolist()==ids and d[\'y\'].tolist()==[r[\'label\'] for r in rr], \'缓存顺序或标签不符\')\n        check(np.allclose(np.linalg.norm(v,axis=1),1,atol=1e-4) and np.allclose(np.linalg.norm(t,axis=1),1,atol=1e-4), \'特征未归一化\')\n        check(np.allclose(x[:,1024:1536],abs(v-t),atol=1e-6) and np.allclose(x[:,1536:],v*t,atol=1e-6), \'特征分段不符\')\n        image_audit = old/\'handoff/manifests/image_audit.jsonl\'\n        check(record(image_audit)==meta[\'image_audit_sha256\'], \'图片审计来源变化\')\n        d[\'rows\']=rr; data[split]=d\n    runpath = old/\'handoff/predictions/train/run.json\'; run = read(runpath); record(runpath)\n    check(run[\'saved_order\']==[\'E\',\'N\',\'C\'] and run[\'manifest_sha256\']==files[\'handoff/manifests/train.jsonl\'], \'Teacher 顺序/来源不符\')\n    logits = {}\n    image_rows = rows(old/\'handoff/manifests/image_audit.jsonl\')\n    image_map = {r[\'sample_id\']:r for r in image_rows}\n    check(len(image_map)==len(image_rows), \'图片审计重复 ID\')\n    for method in [\'jev\',\'base\']:\n        p = old/f\'handoff/predictions/train/{method}.jsonl\'; rr=rows(p); digest=record(p)\n        if method==\'jev\': check(digest==lock[\'teacher_train_sha256\'], \'Jev 缓存与训练锁不符\')\n        byid={r[\'sample_id\']:r for r in rr}\n        check(len(byid)==len(rr)==6000 and set(byid)==set(data[\'train\'][\'ids\']), \'教师缓存缺失/重复/多余样本\')\n        ordered=[byid[r[\'sample_id\']] for r in data[\'train\'][\'rows\']]\n        for r, pred in zip(data[\'train\'][\'rows\'],ordered):\n            check(pred[\'model\']==method and all(pred[k]==r[k] for k in [\'label\',\'image_id\',\'image_file\',\'hypothesis\',\'split\']), \'教师缓存内容错配\')\n            check(pred[\'image_sha256\']==image_map[r[\'sample_id\']][\'file_sha256\'], \'教师图片哈希错配\')\n        z=np.asarray([r[\'logits\'] for r in ordered],dtype=np.float64)\n        check(z.shape==(6000,3) and np.isfinite(z).all(), \'教师 logits 异常\'); logits[method]=z\n    calpath=old/\'handoff/results/calibration.json\'; cal=read(calpath)\n    check(record(calpath)==lock[\'calibration_sha256\'] and cal[\'labels\']==[\'E\',\'N\',\'C\'], \'校准文件不符\')\n    params=cal[\'models\'][\'jev\'][\'selected_params\']\n    check(np.isfinite([params[\'T\'],params[\'b\']]).all() and params[\'T\']>0, \'校准参数异常\')\n    package_manifest=old.parent/\'文件校验清单.json\'\n    if package_manifest.exists():\n        hashes=read(package_manifest)\n        for name,digest in files.items(): check(hashes[\'reused_v1/\'+name]==digest, \'成果包总清单不符：\'+name)\n    else:\n        raise ValueError(\'需要10月13日成果包的文件校验清单.json，以验证 base 缓存来源。请指定完整成果包。\')\n    report=dict(status=\'PASS\', source=str(old), input_sha256=files, clip_revision=REV,\n                counts={s:len(d[\'y\']) for s,d in data.items()}, label_order=[\'E\',\'N\',\'C\'],\n                scope=\'历史缓存完整性与来源；未重新读取原图或重新推理 CLIP/Teacher\', calibration=params)\n    return dict(old=old, data=data, logits=logits, params=params, report=report)\n\ndef seed_all(seed):\n    import torch\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n\ndef make_head(method):\n    import torch.nn as nn\n    d,h=(512,2048) if method in [\'CE_image\',\'CE_text\'] else (2048,512)\n    return nn.Sequential(nn.Linear(d,h),nn.ReLU(),nn.Dropout(.1),nn.Linear(h,3))\n\ndef select_features(x, method):\n    return x[:,:512] if method==\'CE_image\' else x[:,512:1024] if method==\'CE_text\' else x\n\ndef target(ctx, method, tau):\n    if method in [\'CE\',\'CE_image\',\'CE_text\']: return None\n    z=ctx[\'logits\'][\'base\' if method==\'KD_base\' else \'jev\'].copy()\n    if method==\'KD_hard\': return z.argmax(1)\n    if method==\'KD_calibrated\': z=(z+np.array([ctx[\'params\'][\'b\'],0,0]))/ctx[\'params\'][\'T\']\n    z=z/tau; z-=z.max(1,keepdims=True); p=np.exp(z); return (p/p.sum(1,keepdims=True)).astype(np.float32)\n\ndef loss_fn(z,y,q,method,alpha,tau):\n    import torch.nn.functional as F\n    ce=F.cross_entropy(z,y)\n    if q is None: return ce\n    kd=F.cross_entropy(z,q) if method==\'KD_hard\' else tau*tau*F.kl_div(F.log_softmax(z/tau,dim=-1),q,reduction=\'batchmean\')\n    return (1-alpha)*ce+alpha*kd\n\ndef benchmark(ctx, out, methods=METHODS, seed=20261010, alpha=.75, tau=2., batch=128, warmup=10, steps=50, repeats=3):\n    import torch\n    check(torch.cuda.is_available(), \'短跑要求 GPU；不自动用 CPU 替代计时。\')\n    check(0<=alpha<=1 and tau>0 and batch>0 and warmup>=1 and steps>=1 and repeats>=1, \'计时参数不合法\')\n    check(len(methods)>0 and len(set(methods))==len(methods) and set(methods)<=set(METHODS), \'未知或重复方法\')\n    out=Path(out); out.mkdir(parents=True, exist_ok=False)\n    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=True\n    env=dict(torch=str(torch.__version__), numpy=np.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0),\n             python=platform.python_version(), cudnn=torch.backends.cudnn.version(), dtype=\'float32\', tf32_matmul=False, tf32_cudnn=True)\n    config=dict(methods=list(methods), seed=seed, alpha=alpha,tau=tau,batch=batch,warmup=warmup,steps=steps,repeats=repeats,\n                scope=\'GPU-resident cached features -> head training step; forward/loss/backward/AdamW; synchronized per step\',\n                excludes=[\'CLIP/Teacher inference\',\'disk read\',\'host-to-device copy\',\'validation\',\'checkpoint I/O\'], runtime_sha256=sha(__file__))\n    save(out/\'protocol.json\',dict(config=config,environment=env,cache_audit=ctx[\'report\']))\n    x=torch.tensor(ctx[\'data\'][\'train\'][\'x\'],device=\'cuda\'); y=torch.tensor(ctx[\'data\'][\'train\'][\'y\'],device=\'cuda\',dtype=torch.long)\n    check(batch<=len(y), \'batch 超过样本数\')\n    # Exact same sequence of full batches for every method and repeat; no tail batch in step timing.\n    g=torch.Generator(device=\'cpu\').manual_seed(seed)\n    batches=[]\n    while len(batches)<warmup+steps:\n        batches.extend(ix for ix in torch.randperm(len(y),generator=g).split(batch) if len(ix)==batch)\n    batches=[ix.to(\'cuda\') for ix in batches[:warmup+steps]]\n    raw=[]; summary=[]\n    for method in methods:\n        qn=target(ctx,method,tau); q=None if qn is None else torch.tensor(qn,device=\'cuda\',dtype=torch.long if method==\'KD_hard\' else torch.float32)\n        xx=select_features(x,method); times=[]; peaks=[]\n        for repeat in range(repeats):\n            seed_all(seed); model=make_head(method).cuda().train(); opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.0001)\n            def step(ix):\n                opt.zero_grad(set_to_none=True)\n                loss=loss_fn(model(xx[ix]),y[ix],None if q is None else q[ix],method,alpha,tau)\n                loss.backward(); opt.step(); return loss\n            for ix in batches[:warmup]: step(ix)\n            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()\n            for j,ix in enumerate(batches[warmup:]):\n                torch.cuda.synchronize(); start=time.perf_counter(); loss=step(ix); torch.cuda.synchronize()\n                ms=(time.perf_counter()-start)*1000\n                value=float(loss.detach().cpu()); check(math.isfinite(value), \'短跑损失非有限\')\n                times.append(ms); raw.append(dict(method=method,repeat=repeat+1,step=j+1,ms=ms,loss=value))\n            peaks.append(torch.cuda.max_memory_allocated()/1024**2)\n            del opt,model; torch.cuda.empty_cache()\n        mean=float(np.mean(times)); median=float(np.median(times))\n        summary.append(dict(method=method,mean_step_ms=mean,median_step_ms=median,p95_step_ms=float(np.percentile(times,95)),\n                            samples_per_second=batch*1000/mean,peak_allocated_MiB=max(peaks),\n                            estimated_30_epochs_3_seeds_train_only_seconds=mean/1000*math.ceil(len(y)/batch)*30*3))\n        save(out/\'partial_results.json\',dict(summary=summary,raw=raw)); print(method, summary[-1],flush=True)\n        del q\n    save(out/\'results.json\',dict(summary=summary,raw=raw,note=\'工程短跑；全预算仅线性粗估，末批按整批估算；不含验证/保存，不能作为正式训练耗时或端到端加速比。\'))\n    with (out/\'timing_summary.csv\').open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(summary[0])); w.writeheader(); w.writerows(summary)\n    return summary\n', 'experiment.py': '"""Paired CE/label-smoothing/KD experiment and student temperature scaling."""\nfrom pathlib import Path\nfrom contextlib import contextmanager\nimport csv, io, json, math, os, platform, time, uuid\nimport numpy as np\nimport cache_audit as base\n\ncheck, sha, read, rows, save = base.check, base.sha, base.read, base.rows, base.save\nDEFAULT = dict(seeds=[20261010,20261011,20261012], methods=[\'CE\',\'LS\',\'KD_raw\'],\n               label_smoothing=.1, alpha=.75, tau=2., epochs=30, batch=128,\n               lr=.001, weight_decay=.0001, temperature_min=.05, temperature_max=20., temperature_points=401)\n\ndef atomic_bytes(p, value):\n    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)\n    tmp=p.with_name(p.name+\'.\'+uuid.uuid4().hex+\'.partial\')\n    with tmp.open(\'xb\') as f: f.write(value); f.flush(); os.fsync(f.fileno())\n    tmp.replace(p)\n\ndef save_torch(p, value):\n    import torch\n    b=io.BytesIO(); torch.save(value,b); atomic_bytes(p,b.getvalue())\n\ndef freeze(p,value):\n    if Path(p).exists(): check(read(p)==value,\'锁定内容发生变化，请使用新 WORK：\'+str(p))\n    else: save(p,value)\n\n@contextmanager\ndef exclusive(work):\n    p=Path(work)/\'ACTIVE.lock\'\n    with p.open(\'x\',encoding=\'utf-8\') as f: json.dump(dict(pid=os.getpid(),note=\'确认旧进程结束后才能手动移除此锁\'),f)\n    try: yield\n    finally: p.unlink()\n\ndef environment(device=\'cuda\'):\n    import torch\n    check(device in [\'cuda\',\'cpu\'],\'device 不合法\')\n    if device==\'cuda\': check(torch.cuda.is_available(),\'请启用 Colab GPU\')\n    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False\n    return dict(torch=str(torch.__version__),numpy=np.__version__,python=platform.python_version(),\n                device=device,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0) if device==\'cuda\' else None,\n                cudnn=torch.backends.cudnn.version(),tf32=False,dtype=\'float32\')\n\ndef prepare(source,work,config=None,device=\'cuda\'):\n    config=json.loads(json.dumps(DEFAULT if config is None else config))\n    check(set(config)==set(DEFAULT),\'配置项不完整\')\n    check(config[\'methods\']==[\'CE\',\'LS\',\'KD_raw\'],\'本对照固定 CE/LS/KD_raw 三组\')\n    check(len(config[\'seeds\'])==3 and len(set(config[\'seeds\']))==3,\'需要三个不同种子\')\n    check(0<config[\'label_smoothing\']<1 and 0<=config[\'alpha\']<=1 and config[\'tau\']>0,\'损失参数异常\')\n    check(config[\'epochs\']>=1 and config[\'batch\']>=1 and config[\'lr\']>0 and config[\'weight_decay\']>=0,\'训练参数异常\')\n    check(0<config[\'temperature_min\']<1<config[\'temperature_max\'] and config[\'temperature_points\']>=3,\'温度网格异常\')\n    source,work=Path(source).resolve(),Path(work).resolve()\n    check(not work.is_relative_to(source) and not source.is_relative_to(work),\'SOURCE/WORK 必须相互独立\')\n    ctx=base.audit(source); work.mkdir(parents=True,exist_ok=True)\n    ctx.update(work=work,config=config,device=device)\n    protocol=dict(kind=\'CE_LS_student_temperature_v1\',config=config,environment=environment(device),\n                  cache=ctx[\'report\'],code={n:sha(Path(__file__).parent/n) for n in [\'experiment.py\',\'cache_audit.py\']},\n                  selection=\'val_select max macro-F1; tie min uncalibrated NLL; tie earliest epoch\',\n                  calibration=\'per checkpoint scalar positive temperature, val_fit NLL grid minimum; no bias\',\n                  evaluation=\'val_select descriptive reused validation; optional historical test already exposed\')\n    freeze(work/\'protocol.json\',protocol); ctx[\'protocol_sha256\']=sha(work/\'protocol.json\')\n    return ctx\n\ndef softmax(z):\n    z=np.asarray(z,dtype=np.float64); z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)\n\ndef nll(z,y):\n    z=np.asarray(z,dtype=np.float64); m=z.max(1)\n    return float(np.mean(m+np.log(np.exp(z-m[:,None]).sum(1))-z[np.arange(len(y)),y]))\n\ndef metrics(z,y):\n    z=np.asarray(z,dtype=np.float64); y=np.asarray(y,dtype=np.int64)\n    check(z.shape==(len(y),3) and len(y)>0 and np.isfinite(z).all(),\'预测异常\')\n    check(np.isin(y,[0,1,2]).all(),\'标签异常\')\n    p=softmax(z); pred=z.argmax(1); cm=np.zeros((3,3),dtype=np.int64); np.add.at(cm,(y,pred),1)\n    f=2*cm.diagonal()/np.maximum(cm.sum(0)+cm.sum(1),1)\n    confidence=p.max(1); correct=pred==y; bins=np.minimum((confidence*10).astype(int),9)\n    ece=sum(float((bins==b).mean())*abs(float(correct[bins==b].mean())-float(confidence[bins==b].mean())) for b in range(10) if np.any(bins==b))\n    return dict(accuracy=float(correct.mean()),macro_f1=float(f.mean()),nll=nll(z,y),\n                brier=float(np.mean(np.sum((p-np.eye(3)[y])**2,axis=1))),ece10=ece,\n                f1_E=float(f[0]),f1_N=float(f[1]),f1_C=float(f[2]))\n\ndef fit_temperature(z,y,cfg):\n    grid=np.unique(np.r_[1.,np.geomspace(cfg[\'temperature_min\'],cfg[\'temperature_max\'],cfg[\'temperature_points\'])])\n    values=[nll(z/t,y) for t in grid]\n    i=min(range(len(grid)),key=lambda i:(values[i],abs(math.log(grid[i]))))\n    return dict(T=float(grid[i]),fit_nll_raw=nll(z,y),fit_nll_calibrated=values[i],\n                at_boundary=i in [0,len(grid)-1],grid=grid.tolist(),nll_grid=values)\n\ndef checked_feature(ctx,split,folder):\n    folder=Path(folder); p=folder/f\'{split}.npz\'; meta=read(folder/f\'{split}_meta.json\')\n    manifest=ctx[\'old\']/f\'handoff/manifests/{split}.jsonl\'; rr=rows(manifest)\n    check(meta[\'npz_sha256\']==sha(p) and meta[\'manifest_sha256\']==sha(manifest),\'新增特征哈希不符\')\n    check(meta[\'revision\']==base.REV and meta[\'frozen_encoder\'] and meta[\'feature_order\']==[\'v\',\'t\',\'abs(v-t)\',\'v*t\'],\'新增特征定义不符\')\n    check(meta[\'image_audit_sha256\']==sha(ctx[\'old\']/\'handoff/manifests/image_audit.jsonl\'),\'图像审计来源变化\')\n    with np.load(p,allow_pickle=False) as a: d={k:a[k].copy() for k in [\'x\',\'y\',\'ids\',\'image_ids\']}\n    x=d[\'x\']; check(x.shape==(len(rr),2048) and x.dtype==np.float32 and np.isfinite(x).all(),\'新增特征异常\')\n    check(d[\'ids\'].tolist()==[r[\'sample_id\'] for r in rr] and d[\'y\'].tolist()==[r[\'label\'] for r in rr] and d[\'image_ids\'].astype(str).tolist()==[str(r[\'image_id\']) for r in rr],\'新增特征 ID/标签错配\')\n    v,t=x[:,:512],x[:,512:1024]\n    check(np.allclose(np.linalg.norm(v,axis=1),1,atol=1e-4) and np.allclose(np.linalg.norm(t,axis=1),1,atol=1e-4),\'新增特征未归一化\')\n    check(np.allclose(x[:,1024:1536],abs(v-t),atol=1e-6) and np.allclose(x[:,1536:],v*t,atol=1e-6),\'新增特征分段不符\')\n    d[\'rows\']=rr; return d\n\ndef feature_data(ctx,split):\n    if split in ctx[\'data\']: return ctx[\'data\'][split]\n    return checked_feature(ctx,split,ctx[\'work\']/\'features\')\n\ndef extract_missing(ctx,images,split=\'val_fit\'):\n    """Only new output cache; reuse requires verified manifest, payload and provenance."""\n    import torch\n    import torch.nn.functional as F\n    import importlib.metadata as md\n    from PIL import Image\n    from transformers import CLIPModel,CLIPProcessor\n    check(split in [\'val_fit\',\'test\'],\'只补齐 val_fit 或 test\')\n    if split==\'test\': check((ctx[\'work\']/\'student_calibration.json\').exists(),\'先锁定校准再准备测试输入\')\n    folder=ctx[\'work\']/\'features\'; folder.mkdir(exist_ok=True)\n    p=folder/f\'{split}.npz\'; mp=folder/f\'{split}_meta.json\'\n    if p.exists() and mp.exists(): return checked_feature(ctx,split,folder)\n    check(ctx[\'device\']==\'cuda\',\'正式特征提取需要 GPU\')\n    check(md.version(\'transformers\')==\'4.57.1\',\'请使用 transformers 4.57.1\')\n    images=Path(images).resolve(); audit_path=ctx[\'old\']/\'handoff/manifests/image_audit.jsonl\'\n    audit={r[\'sample_id\']:r for r in rows(audit_path)}\n    manifest=ctx[\'old\']/f\'handoff/manifests/{split}.jsonl\'; rr=rows(manifest)\n    known=ctx[\'data\'][\'val_select\'][\'rows\'][:32]\n    def image_path(r):\n        path=(images/r[\'image_file\']).resolve()\n        check(path.is_relative_to(images),\'图片路径越界\')\n        check(path.is_file() and sha(path)==audit[r[\'sample_id\']][\'file_sha256\'],\'图片缺失/哈希不符：\'+str(path))\n        return path\n    for r in rr+known: image_path(r)\n    with exclusive(ctx[\'work\']):\n        model=CLIPModel.from_pretrained(\'openai/clip-vit-base-patch32\',revision=base.REV).to(\'cuda\',dtype=torch.float32).eval()\n        processor=CLIPProcessor.from_pretrained(\'openai/clip-vit-base-patch32\',revision=base.REV)\n        def encode(batch):\n            ims=[]\n            for r in batch:\n                with Image.open(image_path(r)) as im: ims.append(im.convert(\'RGB\'))\n            inp=processor(text=[r[\'hypothesis\'] for r in batch],images=ims,padding=True,truncation=True,max_length=77,return_tensors=\'pt\').to(\'cuda\')\n            with torch.inference_mode():\n                output=model(**inp); v=F.normalize(output.image_embeds.float(),dim=-1); t=F.normalize(output.text_embeds.float(),dim=-1)\n                return torch.cat([v,t,abs(v-t),v*t],dim=-1).cpu().numpy()\n        try:\n            delta=float(abs(encode(known)-ctx[\'data\'][\'val_select\'][\'x\'][:32]).max())\n            check(delta<=1e-4,f\'与旧特征兼容检查失败：max abs={delta}；不要放宽阈值后混用。\')\n            provenance=dict(environment=environment(ctx[\'device\']),transformers=md.version(\'transformers\'),revision=base.REV)\n            freeze(folder/\'extraction_environment.json\',provenance)\n            values=[]; truncated=[]\n            for start in range(0,len(rr),32):\n                batch=rr[start:start+32]; values.append(encode(batch))\n                truncated += [r[\'sample_id\'] for r in batch if len(processor.tokenizer(r[\'hypothesis\'])[\'input_ids\'])>77]\n            b=io.BytesIO(); np.savez_compressed(b,x=np.concatenate(values),y=[r[\'label\'] for r in rr],ids=[r[\'sample_id\'] for r in rr],image_ids=[r[\'image_id\'] for r in rr])\n            atomic_bytes(p,b.getvalue()); save(mp,dict(npz_sha256=sha(p),manifest_sha256=sha(manifest),revision=base.REV,\n                 frozen_encoder=True,feature_order=[\'v\',\'t\',\'abs(v-t)\',\'v*t\'],image_audit_sha256=sha(audit_path),\n                 compatibility_max_abs=delta,truncated_ids=truncated,extraction_environment=provenance))\n        finally:\n            del model; torch.cuda.empty_cache()\n    return checked_feature(ctx,split,folder)\n\ndef training_loss(z,y,q,method,cfg):\n    import torch.nn.functional as F\n    if method==\'LS\': return F.cross_entropy(z,y,label_smoothing=cfg[\'label_smoothing\'])\n    ce=F.cross_entropy(z,y)\n    if method==\'CE\': return ce\n    check(method in [\'KD_raw\',\'KD_calibrated\'],\'未知方法\')\n    return (1-cfg[\'alpha\'])*ce+cfg[\'alpha\']*cfg[\'tau\']**2*F.kl_div(F.log_softmax(z/cfg[\'tau\'],dim=-1),q,reduction=\'batchmean\')\n\ndef predict(model,x,device):\n    import torch\n    model.eval(); zs=[]\n    with torch.inference_mode():\n        for start in range(0,len(x),256): zs.append(model(torch.as_tensor(x[start:start+256],device=device)).cpu().numpy())\n    return np.concatenate(zs)\n\ndef train_one(ctx,method,seed,stop_after_epoch=None):\n    """Epoch-atomic resume. Epoch-specific RNG makes resumed dropout/shuffle identical."""\n    import torch\n    cfg=ctx[\'config\']; device=ctx[\'device\']; folder=ctx[\'work\']/f\'runs/{method}_{seed}\'; folder.mkdir(parents=True,exist_ok=True)\n    statepath=folder/\'state.pt\'; donepath=folder/\'done.json\'\n    identity=dict(protocol_sha256=ctx[\'protocol_sha256\'],method=method,seed=seed)\n    if donepath.exists():\n        done=read(donepath); check(done[\'identity\']==identity and done[\'checkpoint_sha256\']==sha(folder/\'best.pt\'),\'已完成 checkpoint 变化\')\n        return done\n    check(not (ctx[\'work\']/\'selection.json\').exists(),\'模型已锁定，不能重新训练\')\n    a=ctx[\'data\'][\'train\']; b=ctx[\'data\'][\'val_select\']; base.seed_all(seed)\n    model=base.make_head(\'CE\').to(device); opt=torch.optim.AdamW(model.parameters(),lr=cfg[\'lr\'],weight_decay=cfg[\'weight_decay\'])\n    x=torch.as_tensor(a[\'x\'],device=device); y=torch.as_tensor(a[\'y\'],dtype=torch.long,device=device)\n    q=torch.tensor(base.target(ctx,method,cfg[\'tau\']),dtype=torch.float32,device=device) if method in [\'KD_raw\',\'KD_calibrated\'] else None\n    history=[]; best=(-float(\'inf\'),-float(\'inf\')); best_state=None; best_epoch=None; start=1\n    if statepath.exists():\n        s=torch.load(statepath,map_location=\'cpu\',weights_only=True); check(s[\'identity\']==identity,\'断点来源不一致\')\n        model.load_state_dict(s[\'model\']); opt.load_state_dict(s[\'optimizer\']); history=s[\'history\']; best=tuple(s[\'best\'])\n        best_state=s[\'best_state\']; best_epoch=s[\'best_epoch\']; start=s[\'epoch\']+1\n    for epoch in range(start,cfg[\'epochs\']+1):\n        base.seed_all(seed+100000*epoch); model.train(); total=0.; begun=time.perf_counter()\n        g=torch.Generator(device=\'cpu\').manual_seed(seed+epoch)\n        for ix in torch.randperm(len(y),generator=g).split(cfg[\'batch\']):\n            ix=ix.to(device); opt.zero_grad(set_to_none=True)\n            loss=training_loss(model(x[ix]),y[ix],None if q is None else q[ix],method,cfg)\n            check(bool(torch.isfinite(loss).item()),\'训练损失非有限\')\n            loss.backward(); opt.step(); total+=float(loss.detach().cpu())*len(ix)\n        z=predict(model,b[\'x\'],device); m=metrics(z,b[\'y\']); key=(m[\'macro_f1\'],-m[\'nll\'])\n        if key>best:\n            best=key; best_epoch=epoch; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n        history.append(dict(epoch=epoch,loss=total/len(y),validation=m,train_and_validation_seconds=time.perf_counter()-begun))\n        save_torch(statepath,dict(identity=identity,epoch=epoch,model=model.state_dict(),optimizer=opt.state_dict(),history=history,\n                                  best=list(best),best_state=best_state,best_epoch=best_epoch))\n        print(method,seed,f\'{epoch}/{cfg["epochs"]}\',f\'val F1={m["macro_f1"]:.4f}\',flush=True)\n        if stop_after_epoch is not None and epoch>=stop_after_epoch and epoch<cfg[\'epochs\']: return None\n    check(best_state is not None and len(history)==cfg[\'epochs\'],\'训练不完整\')\n    save_torch(folder/\'best.pt\',best_state); save(folder/\'history.json\',history)\n    done=dict(identity=identity,method=method,seed=seed,best_epoch=best_epoch,validation_macro_f1=best[0],validation_nll=-best[1],\n              checkpoint=str((folder/\'best.pt\').relative_to(ctx[\'work\'])),checkpoint_sha256=sha(folder/\'best.pt\'),history_sha256=sha(folder/\'history.json\'))\n    save(donepath,done); return done\n\ndef train_all(ctx):\n    environment(ctx[\'device\']); selected=[]\n    with exclusive(ctx[\'work\']):\n        for seed in ctx[\'config\'][\'seeds\']:\n            for method in ctx[\'config\'][\'methods\']: selected.append(train_one(ctx,method,seed))\n        freeze(ctx[\'work\']/\'selection.json\',dict(protocol_sha256=ctx[\'protocol_sha256\'],models=selected))\n    return selected\n\ndef selected_models(ctx):\n    selection=read(ctx[\'work\']/\'selection.json\'); check(selection[\'protocol_sha256\']==ctx[\'protocol_sha256\'],\'选模协议变化\')\n    expected={(m,s) for m in ctx[\'config\'][\'methods\'] for s in ctx[\'config\'][\'seeds\']}\n    check(len(selection[\'models\'])==len(expected) and {(m[\'method\'],m[\'seed\']) for m in selection[\'models\']}==expected,\'选模不完整\')\n    for m in selection[\'models\']: check(sha(ctx[\'work\']/m[\'checkpoint\'])==m[\'checkpoint_sha256\'],\'checkpoint 损坏\')\n    return selection[\'models\']\n\ndef model_logits(ctx,model_info,data):\n    import torch\n    model=base.make_head(\'CE\').to(ctx[\'device\'])\n    model.load_state_dict(torch.load(ctx[\'work\']/model_info[\'checkpoint\'],map_location=\'cpu\',weights_only=True))\n    return predict(model,data[\'x\'],ctx[\'device\'])\n\ndef calibrate(ctx):\n    data=feature_data(ctx,\'val_fit\'); models=selected_models(ctx); path=ctx[\'work\']/\'student_calibration.json\'\n    inputs=dict(protocol_sha256=ctx[\'protocol_sha256\'],selection_sha256=sha(ctx[\'work\']/\'selection.json\'),\n                val_fit_npz_sha256=sha(ctx[\'work\']/\'features/val_fit.npz\'),val_fit_meta_sha256=sha(ctx[\'work\']/\'features/val_fit_meta.json\'))\n    if path.exists():\n        result=read(path); check(result[\'inputs\']==inputs,\'校准输入变化\'); return result\n    with exclusive(ctx[\'work\']):\n        result=[]\n        for m in models:\n            z=model_logits(ctx,m,data); fit=fit_temperature(z,data[\'y\'],ctx[\'config\'])\n            check(np.array_equal(z.argmax(1),(z/fit[\'T\']).argmax(1)),\'正温度不应改变 argmax\')\n            result.append(dict(method=m[\'method\'],seed=m[\'seed\'],checkpoint_sha256=m[\'checkpoint_sha256\'],**fit))\n            print(\'temperature\',m[\'method\'],m[\'seed\'],fit[\'T\'],\'boundary=\',fit[\'at_boundary\'],flush=True)\n        value=dict(inputs=inputs,models=result,scope=\'fit NLL on val_fit; this fit loss is not evaluation performance\')\n        freeze(path,value)\n    return value\n\ndef csvwrite(p,records):\n    if not records: return\n    b=io.StringIO(newline=\'\'); w=csv.DictWriter(b,fieldnames=list(records[0])); w.writeheader(); w.writerows(records)\n    atomic_bytes(p,b.getvalue().encode(\'utf-8-sig\'))\n\ndef report(ctx,split=\'val_select\',allow_historical_test=False):\n    check(split in [\'val_select\',\'test\'],\'报告划分不合法\')\n    check(split!=\'test\' or allow_historical_test,\'历史测试复评开关未开启\')\n    models=selected_models(ctx); calibration=calibrate(ctx); data=feature_data(ctx,split)\n    out=ctx[\'work\']/f\'reports/{split}\'; out.mkdir(parents=True,exist_ok=True)\n    inputs=dict(protocol_sha256=ctx[\'protocol_sha256\'],selection_sha256=sha(ctx[\'work\']/\'selection.json\'),calibration_sha256=sha(ctx[\'work\']/\'student_calibration.json\'),\n                feature_sha256=sha((ctx[\'old\'] if split==\'val_select\' else ctx[\'work\'])/f\'features/{split}.npz\'))\n    if (out/\'COMPLETE.json\').exists():\n        done=read(out/\'COMPLETE.json\'); check(done[\'inputs\']==inputs,\'已完成报告输入改变\')\n        for name,h in done[\'files\'].items(): check(sha(out/name)==h,\'报告文件损坏\')\n        return out\n    with exclusive(ctx[\'work\']):\n        values=[]; predictions=[]\n        for m in models:\n            fit=next(c for c in calibration[\'models\'] if c[\'method\']==m[\'method\'] and c[\'seed\']==m[\'seed\'])\n            check(fit[\'checkpoint_sha256\']==m[\'checkpoint_sha256\'],\'校准与模型错配\')\n            z=model_logits(ctx,m,data)\n            for variant,T in [(\'raw\',1.),(\'temperature\',fit[\'T\'])]:\n                zz=z.astype(np.float64)/T; p=softmax(zz)\n                values.append(dict(method=m[\'method\'],seed=m[\'seed\'],variant=variant,T=T,**metrics(zz,data[\'y\'])))\n                for i,sid in enumerate(data[\'ids\']):\n                    predictions.append(dict(sample_id=str(sid),image_id=str(data[\'image_ids\'][i]),label=int(data[\'y\'][i]),method=m[\'method\'],seed=m[\'seed\'],variant=variant,\n                                            logit_E=float(zz[i,0]),logit_N=float(zz[i,1]),logit_C=float(zz[i,2]),p_E=float(p[i,0]),p_N=float(p[i,1]),p_C=float(p[i,2]),prediction=int(zz[i].argmax())))\n            a,b=values[-2:]; check(a[\'accuracy\']==b[\'accuracy\'] and a[\'macro_f1\']==b[\'macro_f1\'],\'温度校准意外改变分类\')\n        metric_names=[\'accuracy\',\'macro_f1\',\'nll\',\'brier\',\'ece10\',\'f1_E\',\'f1_N\',\'f1_C\']; summary=[]; deltas=[]\n        for method in ctx[\'config\'][\'methods\']:\n            for variant in [\'raw\',\'temperature\']:\n                group=[r for r in values if r[\'method\']==method and r[\'variant\']==variant]\n                summary.append(dict(method=method,variant=variant,**{k+suffix:float(fn([r[k] for r in group])) for k in metric_names for suffix,fn in [(\'_mean\',np.mean),(\'_seed_sd\',lambda a:np.std(a,ddof=1))]}))\n        def get(method,seed,variant): return next(r for r in values if r[\'method\']==method and r[\'seed\']==seed and r[\'variant\']==variant)\n        for seed in ctx[\'config\'][\'seeds\']:\n            for variant in [\'raw\',\'temperature\']:\n                for a,b in [(\'LS\',\'CE\'),(\'KD_raw\',\'CE\'),(\'KD_raw\',\'LS\')]:\n                    ra,rb=get(a,seed,variant),get(b,seed,variant)\n                    deltas.append(dict(comparison=a+\' minus \'+b,seed=seed,variant=variant,**{k:ra[k]-rb[k] for k in metric_names}))\n            for method in ctx[\'config\'][\'methods\']:\n                ra,rb=get(method,seed,\'temperature\'),get(method,seed,\'raw\')\n                deltas.append(dict(comparison=method+\' temperature minus raw\',seed=seed,variant=\'postcalibration\',**{k:ra[k]-rb[k] for k in metric_names}))\n        csvwrite(out/\'per_seed.csv\',values); csvwrite(out/\'summary.csv\',summary); csvwrite(out/\'paired_deltas.csv\',deltas); csvwrite(out/\'predictions.csv\',predictions)\n        save(out/\'metrics.json\',dict(inputs=inputs,per_seed=values,summary=summary,paired_deltas=deltas))\n        note=(\'验证选择集，已用于 checkpoint 选择，仅作开发性描述。\' if split==\'val_select\' else \'历史测试集已被查看；本轮复评不能称为全新独立确认。\')\n        text=\'# CE / 标签平滑 / 学生后校准\\n\\n\'+note+\'\\n\\n所有准确率、F1、ECE 用 0–1 原值；差值也是原值。seed_sd 为三个种子之间的样本标准差，不是置信区间。NLL/Brier/ECE 越低越好。\\n\\n温度只改变概率，不改变分类；不保证评价集的概率指标一定改善。边界温度见 student_calibration.json，不据测试结果扩网格。\\n\\n|方法|版本|Macro-F1 均值|NLL 均值|Brier 均值|ECE 均值|\\n|---|---|---:|---:|---:|---:|\\n\'\n        for r in summary: text+=f"|{r[\'method\']}|{r[\'variant\']}|{r[\'macro_f1_mean\']:.4f}|{r[\'nll_mean\']:.4f}|{r[\'brier_mean\']:.4f}|{r[\'ece10_mean\']:.4f}|\\n"\n        atomic_bytes(out/\'README.md\',text.encode(\'utf-8\'))\n        names=[\'per_seed.csv\',\'summary.csv\',\'paired_deltas.csv\',\'predictions.csv\',\'metrics.json\',\'README.md\']\n        save(out/\'COMPLETE.json\',dict(inputs=inputs,files={n:sha(out/n) for n in names}))\n    return out\n', 'stage2.py': '"""Stage2 Hazel: six three-seed KD configurations, frozen teacher calibration."""\nfrom pathlib import Path, PurePosixPath\nimport copy, csv, hashlib, io, json, zipfile, uuid, time\nimport numpy as np\nimport experiment as e\n\nSEEDS=[20261010,20261011,20261012]\nALPHAS=[.25,.5,.75]\nMETHODS=[\'KD_raw\',\'KD_calibrated\']\nMETRICS=[\'macro_f1\',\'accuracy\',\'f1_E\',\'f1_N\',\'f1_C\',\'nll\',\'ece10\',\'brier\']\nCAL_SHA=\'8745dfe1110291ff54abb45362bb922cf13cd9c1bbbd242baa83ff4b86299d26\'\n\ndef table(path,records,fields=None):\n    fields=fields or list(dict.fromkeys(k for r in records for k in r))\n    if not fields: fields=[\'status\']\n    out=io.StringIO(newline=\'\'); w=csv.DictWriter(out,fieldnames=fields); w.writeheader(); w.writerows(records)\n    e.atomic_bytes(path,out.getvalue().encode(\'utf-8-sig\'))\n\ndef readcsv(path):\n    with Path(path).open(encoding=\'utf-8-sig\',newline=\'\') as f: return list(csv.DictReader(f))\n\ndef materialize(source,dest,marker):\n    """Read a directory or safely unpack only this experiment\'s artifacts, no executable imports."""\n    source=Path(source).resolve(); dest=Path(dest).resolve()\n    if source.is_dir():\n        candidates=[source]+[p.parent for p in source.glob(\'*/\'+marker)]\n        matches=list(set(p for p in candidates if (p/marker).is_file()))\n        e.check(len(matches)==1,\'未唯一找到输入 \'+marker)\n        return matches[0]\n    e.check(source.is_file() and zipfile.is_zipfile(source),\'输入须为目录或 ZIP：\'+str(source))\n    with zipfile.ZipFile(source) as z:\n        names=z.namelist(); e.check(len(names)==len(set(names)),\'ZIP 存在重复成员\')\n        matches=[n for n in names if n==marker or n.endswith(\'/\'+marker)]\n        e.check(len(matches)==1,\'ZIP 中未唯一找到 \'+marker)\n        prefix=matches[0][:-len(marker)]\n        for info in z.infolist():\n            if not info.filename.startswith(prefix) or info.is_dir(): continue\n            rel=info.filename[len(prefix):]; parts=PurePosixPath(rel).parts\n            e.check(\'\\\\\' not in rel and \':\' not in rel and \'..\' not in parts and not rel.startswith(\'/\'),\'不安全 ZIP 路径\')\n            # Exclude feature caches, optimizer states, images, and test reports from import.\n            if rel.startswith((\'features/\',\'prior_inputs/\',\'reports/test/\')) or info.filename.endswith((\'state.pt\',\'.zip\')): continue\n            if Path(rel).suffix not in [\'.json\',\'.csv\',\'.py\',\'.pt\',\'.md\']: continue\n            p=(dest/rel).resolve(); e.check(p.is_relative_to(dest),\'ZIP 路径越界\')\n            payload=z.read(info)\n            if p.exists(): e.check(p.read_bytes()==payload,\'已导入文件变化：\'+rel)\n            else: e.atomic_bytes(p,payload)\n    return dest\n\ndef teacher(ctx,receipt):\n    receipt=Path(receipt); cp=receipt/\'teacher_calibration_stage2.json\'; c=e.read(cp)\n    e.check(e.sha(cp)==CAL_SHA,\'不是本次交付的新教师校准文件，拒绝历史参数或改写参数\')\n    hashes=e.read(receipt/\'FILE_SHA256.json\')\n    for name in [\'teacher_calibration_stage2.json\',\'protocol_LOCKED.json\']:\n        e.check(e.sha(receipt/name)==hashes[name],\'教师交付文件哈希不符\')\n    e.check(c[\'protocol_sha256\']==e.sha(receipt/\'protocol_LOCKED.json\'),\'教师协议错配\')\n    e.check(c[\'schema\']==\'stage2_teacher_devcal_v1\' and c[\'labels\']==[\'E\',\'N\',\'C\'],\'教师类别/格式不符\')\n    e.check(c[\'formula\']==\'(raw_logits + [b,0,0]) / T\' and c[\'fit_split\']==\'val_fit\' and c[\'fit_n\']==400,\'校准公式或拟合集不符\')\n    for key,rel in [(\'train_manifest\',\'handoff/manifests/train.jsonl\'),(\'jev_train\',\'handoff/predictions/train/jev.jsonl\'),(\'base_train\',\'handoff/predictions/train/base.jsonl\')]:\n        e.check(e.sha(ctx[\'old\']/rel)==c[\'expected_training_inputs\'][key][\'sha256\'],\'教师交付与训练缓存版本不一致：\'+key)\n    for name,rel in [(\'val_fit.jsonl\',\'handoff/manifests/val_fit.jsonl\'),(\'jev.jsonl\',\'handoff/predictions/val_fit/jev.jsonl\'),(\'teacher_run.json\',\'handoff/predictions/val_fit/run.json\')]:\n        e.check(e.sha(ctx[\'old\']/rel)==c[\'input_sha256\'][name],\'教师拟合来源不一致：\'+name)\n    run=e.read(ctx[\'old\']/\'handoff/predictions/train/run.json\')\n    e.check(run[\'revisions\']==c[\'teacher_revisions\'] and run[\'dtype\']==c[\'teacher_dtype\'],\'教师模型或精度不一致\')\n    p=c[\'models\'][\'jev\'][\'selected_params\']\n    e.check(np.isfinite([p[\'T\'],p[\'b\']]).all() and p[\'T\']>0,\'T/b 非法\')\n    return c,dict(T=float(p[\'T\']),b=float(p[\'b\']))\n\ndef load_day2(ctx,root):\n    pp=e.read(root/\'protocol.json\'); lock=e.read(root/\'common_lr.json\'); sel=e.read(root/\'supervised_selection.json\')\n    ph=e.sha(root/\'protocol.json\')\n    e.check(pp[\'kind\']==\'CE_3LR_then_2LS_v1\',\'DAY2 必须是 CE三学习率/两种LS 补充输出\')\n    e.check(pp[\'input_sha256\']==ctx[\'report\'][\'input_sha256\'],\'Day2 与本次缓存不一致\')\n    e.check(lock[\'parent_sha256\']==ph and sel[\'parent_sha256\']==ph and sel[\'common_lr_sha256\']==e.sha(root/\'common_lr.json\'),\'Day2 选择锁损坏\')\n    e.check(lock[\'lr\']==sel[\'lr\']==.0003,\'Day2 公共学习率不是0.0003，请核对输入目录\')\n    for name,h in pp[\'code\'].items():\n        p=(root/\'code\'/name).resolve()\n        e.check(p.is_relative_to((root/\'code\').resolve()) and e.sha(p)==h,\'Day2 代码快照损坏\')\n    cfg=copy.deepcopy(pp[\'base_config\'])\n    e.check(cfg[\'seeds\']==SEEDS and cfg[\'epochs\']==30 and cfg[\'batch\']==128 and cfg[\'weight_decay\']==.0001,\'Day2 预算或种子与既定方案不符\')\n    expected={(lr,s) for lr in pp[\'lrs\'] for s in SEEDS}\n    e.check(len(lock[\'models\'])==9 and {(m[\'lr\'],m[\'seed\']) for m in lock[\'models\']}==expected,\'Day2 学习率比较不完整\')\n    choices=[]\n    for lr in pp[\'lrs\']:\n        rr=[m for m in lock[\'models\'] if m[\'lr\']==lr]\n        choices.append((float(np.mean([m[\'validation_macro_f1\'] for m in rr])),float(np.mean([m[\'validation_nll\'] for m in rr])),lr))\n    e.check(min(choices,key=lambda x:(-x[0],x[1],x[2]))[2]==.0003,\'Day2 学习率选择记录不一致\')\n    e.check(len(sel[\'all_15_models\'])==15 and len({(m[\'method\'],m[\'lr\'],m[\'epsilon\'],m[\'seed\']) for m in sel[\'all_15_models\']})==15,\'Day2 15条件清单不完整\')\n    for m in sel[\'all_15_models\']:\n        p=(root/m[\'checkpoint\']).resolve(); e.check(p.is_relative_to(root.resolve()),\'Day2 checkpoint 越界\')\n        e.check(e.sha(p)==m[\'checkpoint_sha256\'] and e.sha(p.parent/\'history.json\')==m[\'history_sha256\'],\'Day2 模型或历史哈希不符\')\n        hist=e.read(p.parent/\'history.json\')\n        e.check([r[\'epoch\'] for r in hist]==list(range(1,cfg[\'epochs\']+1)),\'Day2 训练预算不完整\')\n        best=max(hist,key=lambda r:(r[\'validation\'][\'macro_f1\'],-r[\'validation\'][\'nll\'],-r[\'epoch\']))\n        e.check(best[\'epoch\']==m[\'best_epoch\'] and best[\'validation\'][\'macro_f1\']==m[\'validation_macro_f1\'] and best[\'validation\'][\'nll\']==m[\'validation_nll\'],\'Day2 checkpoint选择规则不一致\')\n    report=root/\'report\'; complete=e.read(report/\'COMPLETE.json\')\n    e.check(complete[\'inputs\'][\'protocol_sha256\']==ph and complete[\'inputs\'][\'selection_sha256\']==e.sha(root/\'supervised_selection.json\') and complete[\'inputs\'][\'calibration_sha256\']==e.sha(root/\'student_calibration.json\'),\'Day2 报告输入错配\')\n    for name,h in complete[\'files\'].items():\n        p=(report/name).resolve(); e.check(p.is_relative_to(report.resolve()) and e.sha(p)==h,\'Day2 报告损坏\')\n    values=[]\n    for r in readcsv(report/\'per_seed.csv\'):\n        row=dict(source=\'Day2\',method=r[\'method\'],alpha=\'\',epsilon=float(r[\'epsilon\']),lr=float(r[\'lr\']),seed=int(r[\'seed\']),variant=r[\'variant\'],T=float(r[\'T\']),**{k:float(r[k]) for k in METRICS})\n        e.check(np.isfinite([row[k] for k in METRICS]).all(),\'Day2 指标非有限\')\n        values.append(row)\n    families=[(\'CE\',0.)]+[(\'LS\',float(v)) for v in pp[\'epsilons\']]\n    expected={(m,eps,s,v) for m,eps in families for s in SEEDS for v in [\'raw\',\'temperature\']}\n    e.check(len(values)==len(expected) and {(r[\'method\'],r[\'epsilon\'],r[\'seed\'],r[\'variant\']) for r in values}==expected,\'Day2 CE/LS及后校准结果不完整或重复\')\n    e.check(all(r[\'lr\']==.0003 for r in values),\'Day2 对照学习率不符\')\n    for r in values:\n        if r[\'variant\']!=\'raw\': continue\n        matches=[m for m in sel[\'models\'] if m[\'method\']==r[\'method\'] and m[\'seed\']==r[\'seed\'] and m[\'epsilon\']==r[\'epsilon\'] and m[\'lr\']==r[\'lr\']]\n        e.check(len(matches)==1,\'Day2 报告模型缺失或重复\')\n        m=matches[0]\n        e.check(np.isclose(r[\'macro_f1\'],m[\'validation_macro_f1\'],rtol=0,atol=1e-12) and np.isclose(r[\'nll\'],m[\'validation_nll\'],rtol=0,atol=1e-12),\'Day2 报告与选定模型分数不符\')\n    cfg.update(methods=METHODS,lr=.0003,tau=2.)\n    return pp,cfg,values\n\ndef snapshot_day2(root,dest):\n    paths=[p for p in root.glob(\'*\') if p.is_file() and p.suffix in [\'.json\',\'.csv\']]\n    paths+=list((root/\'report\').glob(\'*\'))+list((root/\'code\').glob(\'*.py\'))\n    sel=e.read(root/\'supervised_selection.json\')\n    for m in sel[\'all_15_models\']:\n        p=root/m[\'checkpoint\']; paths += [p,p.parent/\'done.json\',p.parent/\'history.json\',p.parent.parent.parent/\'protocol.json\']\n    files={}\n    for p in set(paths):\n        if not p.is_file(): continue\n        rel=p.relative_to(root).as_posix(); target=dest/rel; digest=e.sha(p)\n        if target.exists(): e.check(e.sha(target)==digest,\'Day2 快照发生变化\')\n        else: e.atomic_bytes(target,p.read_bytes())\n        files[rel]=digest\n    return files\n\ndef prepare(source,day2,receipt,work,legacy=None,device=\'cuda\'):\n    source,day2,receipt,work=map(lambda p:Path(p).resolve(),[source,day2,receipt,work])\n    for p in [source,day2,receipt]+([Path(legacy).resolve()] if legacy else []):\n        e.check(not work.is_relative_to(p) and not p.is_relative_to(work),\'WORK 须与所有输入独立\')\n    ctx=e.base.audit(source); work.mkdir(parents=True,exist_ok=True)\n    prior=materialize(day2,work/\'imported_day2\',\'common_lr.json\')\n    c,params=teacher(ctx,receipt); pp,cfg,baseline=load_day2(ctx,prior)\n    env=e.environment(device)\n    expected=pp[\'environment\']\n    differences={k:dict(day2=expected.get(k),current=env.get(k)) for k in set(expected)|set(env) if k not in expected or k not in env or expected[k]!=env[k]}\n    allowed={\'python\':dict(day2=\'3.13.15\',current=\'3.13.16\')}\n    e.check(not differences or differences==allowed,\'Day2 环境仍有未允许的差异：\'+json.dumps(differences,ensure_ascii=False))\n    env_comparison=dict(day2_environment=expected,current_environment=env,differences=differences,\n        policy=\'exact match except documented Python 3.13.15 to 3.13.16 patch update; not a bitwise equivalence claim\')\n    if differences: print(\'已记录 Python 补丁升级 3.13.15 → 3.13.16；其余环境一致。\',flush=True)\n    files=snapshot_day2(prior,work/\'Day2_preserved\')\n    for name in [\'teacher_calibration_stage2.json\',\'protocol_LOCKED.json\',\'FILE_SHA256.json\']:\n        dest=work/\'teacher_receipt\'/name; payload=(receipt/name).read_bytes()\n        if dest.exists(): e.check(dest.read_bytes()==payload,\'教师快照变化\')\n        else: e.atomic_bytes(dest,payload)\n    legacy_files={}\n    if legacy:\n        old=materialize(legacy,work/\'imported_legacy\',\'selection.json\')\n        oldp=e.read(old/\'protocol.json\')\n        e.check(oldp[\'config\'][\'lr\']==.001,\'补充 KD 必须是旧 lr=.001 实验\')\n        # Preserve available old validation reports, never score them again or include in 18.\n        for p in [old/\'protocol.json\',old/\'selection.json\',old/\'student_calibration.json\']+list((old/\'reports/val_select\').glob(\'*\')):\n            if p.is_file():\n                rel=p.relative_to(old).as_posix(); dst=work/\'legacy_lr0001\'/rel\n                if dst.exists(): e.check(e.sha(dst)==e.sha(p),\'旧补充材料变化\')\n                else: e.atomic_bytes(dst,p.read_bytes())\n                legacy_files[rel]=e.sha(p)\n    protocol=dict(kind=\'Stage2_Hazel_18KD_v1\',config=cfg,alphas=ALPHAS,methods=METHODS,seeds=SEEDS,\n        environment_comparison=env_comparison,environment=env,cache=ctx[\'report\'],teacher_calibration_sha256=e.sha(receipt/\'teacher_calibration_stage2.json\'),teacher_params=params,\n        day2_files=files,legacy_files=legacy_files,legacy_status=\'provided_supplement_only\' if legacy else \'not_provided_old_results_remain_at_source\',\n        code={n:e.sha(Path(__file__).parent/n) for n in [\'cache_audit.py\',\'experiment.py\',\'stage2.py\']},\n        selection=\'val_select max raw Macro-F1, min raw NLL, earliest epoch\',\n        student_temperature=\'KL uses log_softmax(student_logits/tau); metrics use unscaled logits; no new postcalibration\',\n        loss=\'(1-alpha)*CE + alpha*tau^2*KL(teacher_target || softmax(student_logits/tau))\',\n        evaluation=\'development val_select only; no test predictions; aggregate only complete three-seed families\')\n    e.freeze(work/\'protocol.json\',protocol)\n    ctx.update(work=work,config=cfg,device=device,params=params,baseline=baseline,protocol=protocol,protocol_sha256=e.sha(work/\'protocol.json\'))\n    return ctx\n\ndef tag(method,alpha): return f\'{method}_alpha{alpha:g}\'\n\ndef child_context(ctx,method,alpha,create=False):\n    folder=ctx[\'work\']/\'groups\'/tag(method,alpha)\n    cfg=dict(ctx[\'config\'],methods=[method],alpha=alpha)\n    value=dict(parent_sha256=ctx[\'protocol_sha256\'],method=method,config=cfg,teacher_params=ctx[\'params\'] if method==\'KD_calibrated\' else dict(T=1.,b=0.))\n    if create:\n        folder.mkdir(parents=True,exist_ok=True); e.freeze(folder/\'protocol.json\',value)\n    if not (folder/\'protocol.json\').exists(): return None\n    e.check(e.read(folder/\'protocol.json\')==value,\'配置锁发生变化\')\n    return dict(ctx,work=folder,config=cfg,params=value[\'teacher_params\'],protocol_sha256=e.sha(folder/\'protocol.json\'))\n\ndef save_predictions(child,m):\n    folder=(child[\'work\']/m[\'checkpoint\']).parent; result=folder/\'result.json\'\n    identity=dict(protocol_sha256=child[\'protocol_sha256\'],checkpoint_sha256=m[\'checkpoint_sha256\'])\n    if result.exists():\n        r=e.read(result); e.check(r[\'identity\']==identity and r[\'predictions_sha256\']==e.sha(folder/\'predictions.csv\'),\'预测文件变化\')\n        return r[\'metrics\']\n    data=child[\'data\'][\'val_select\']; z=e.model_logits(child,m,data); probs=e.softmax(z)\n    record=dict(source=\'Stage2_KD\',method=m[\'method\'],alpha=child[\'config\'][\'alpha\'],epsilon=\'\',lr=child[\'config\'][\'lr\'],seed=m[\'seed\'],variant=\'raw\',T=1.,best_epoch=m[\'best_epoch\'],**e.metrics(z,data[\'y\']))\n    preds=[dict(sample_id=str(sid),label=int(data[\'y\'][i]),prediction=int(z[i].argmax()),logit_E=float(z[i,0]),logit_N=float(z[i,1]),logit_C=float(z[i,2]),p_E=float(probs[i,0]),p_N=float(probs[i,1]),p_C=float(probs[i,2])) for i,sid in enumerate(data[\'ids\'])]\n    table(folder/\'predictions.csv\',preds)\n    e.freeze(result,dict(identity=identity,metrics=record,predictions_sha256=e.sha(folder/\'predictions.csv\')))\n    return record\n\ndef inventory(ctx):\n    rows=[]; values=[]; families={}\n    for alpha in ALPHAS:\n        for method in METHODS:\n            child=child_context(ctx,method,alpha); rr=[]\n            for seed in SEEDS:\n                folder=ctx[\'work\']/\'groups\'/tag(method,alpha)/f\'runs/{method}_{seed}\'\n                item=dict(method=method,alpha=alpha,seed=seed,lr=.0003,status=\'missing\',checkpoint=\'\',best_epoch=\'\')\n                if (folder/\'state.pt\').exists(): item[\'status\']=\'in_progress\'\n                if (folder/\'done.json\').exists():\n                    e.check(child is not None,\'训练完成但缺协议\')\n                    m=e.read(folder/\'done.json\')\n                    e.check(m[\'identity\']==dict(protocol_sha256=child[\'protocol_sha256\'],method=method,seed=seed),\'模型身份错误\')\n                    e.check(e.sha(folder/\'best.pt\')==m[\'checkpoint_sha256\'] and e.sha(folder/\'history.json\')==m[\'history_sha256\'],\'checkpoint/历史损坏\')\n                    item.update(status=\'trained_pending_predictions\',checkpoint=(folder/\'best.pt\').relative_to(ctx[\'work\']).as_posix(),best_epoch=m[\'best_epoch\'])\n                    if (folder/\'result.json\').exists():\n                        r=e.read(folder/\'result.json\')\n                        e.check(r[\'identity\']==dict(protocol_sha256=child[\'protocol_sha256\'],checkpoint_sha256=m[\'checkpoint_sha256\']) and r[\'predictions_sha256\']==e.sha(folder/\'predictions.csv\'),\'结果损坏\')\n                        rr.append(r[\'metrics\']); item[\'status\']=\'complete\'\n                rows.append(item)\n            complete=len(rr)==3 and {r[\'seed\'] for r in rr}==set(SEEDS)\n            for r in rr: values.append(dict(r,eligible_for_comparison=complete))\n            if complete: families[(method,alpha)]=rr\n    return rows,values,families\n\ndef run_groups(ctx,max_new_groups=6):\n    e.check(isinstance(max_new_groups,int) and 0<=max_new_groups<=6,\'MAX_NEW_GROUPS 只能为0到6整数\')\n    e.check(e.environment(ctx[\'device\'])==ctx[\'protocol\'][\'environment\'],\'会话环境变化\')\n    started=0\n    with e.exclusive(ctx[\'work\']):\n        for alpha in ALPHAS:\n            for method in METHODS:\n                _,_,complete=inventory(ctx)\n                if (method,alpha) in complete: continue\n                if started>=max_new_groups: return\n                started+=1; child=child_context(ctx,method,alpha,create=True)\n                try:\n                    models=[]\n                    for seed in SEEDS:\n                        m=e.train_one(child,method,seed); save_predictions(child,m); models.append(m)\n                    e.freeze(child[\'work\']/\'selection.json\',dict(protocol_sha256=child[\'protocol_sha256\'],models=models))\n                    print(\'完整三种子配置完成：\',tag(method,alpha),flush=True)\n                except (KeyboardInterrupt,RuntimeError) as ex:\n                    e.save(child[\'work\']/f\'interruption_{uuid.uuid4().hex[:8]}.json\',dict(type=type(ex).__name__,message=str(ex),time=time.time()))\n                    if isinstance(ex,KeyboardInterrupt) or \'out of memory\' in str(ex).lower():\n                        print(\'已停止新配置；保留断点，缺项会写入报告。\',flush=True)\n                        return\n                    raise\n\ndef report(ctx):\n    inv,kd,families=inventory(ctx); out=ctx[\'work\']/\'report\'; out.mkdir(exist_ok=True)\n    day2=[dict(r,eligible_for_comparison=True) for r in ctx[\'baseline\']]\n    allrows=day2+kd; summary=[]\n    grouped={}\n    for r in allrows:\n        if not r[\'eligible_for_comparison\']: continue\n        key=tuple(r[k] for k in [\'source\',\'method\',\'alpha\',\'epsilon\',\'lr\',\'variant\'])\n        grouped.setdefault(key,[]).append(r)\n    for key,rr in grouped.items():\n        e.check(len(rr)==3 and {r[\'seed\'] for r in rr}==set(SEEDS),\'汇总必须为完整3种子\')\n        summary.append(dict(zip([\'source\',\'method\',\'alpha\',\'epsilon\',\'lr\',\'variant\'],key),n_seeds=3,\n            **{k+\'_mean\':float(np.mean([r[k] for r in rr])) for k in METRICS},\n            **{k+\'_seed_sd\':float(np.std([r[k] for r in rr],ddof=1)) for k in METRICS}))\n    deltas=[]\n    def pair(a,b,label,category):\n        aa={r[\'seed\']:r for r in a}; bb={r[\'seed\']:r for r in b}\n        e.check(set(aa)==set(bb)==set(SEEDS),\'配对必须包含3种子\')\n        for seed in SEEDS:\n            deltas.append(dict(category=category,comparison=label,seed=seed,**{k:aa[seed][k]-bb[seed][k] for k in METRICS}))\n    ce=[r for r in day2 if r[\'method\']==\'CE\' and r[\'variant\']==\'raw\']\n    for (method,alpha),rr in families.items(): pair(rr,ce,f\'{tag(method,alpha)} minus CE_lr0.0003\',\'KD_vs_CE\')\n    for alpha in ALPHAS:\n        if all((m,alpha) in families for m in METHODS):\n            pair(families[(\'KD_calibrated\',alpha)],families[(\'KD_raw\',alpha)],f\'calibrated minus raw alpha={alpha:g}\',\'calibrated_vs_raw\')\n    for method in METHODS:\n        for i,high in enumerate(ALPHAS):\n            for low in ALPHAS[:i]:\n                if (method,high) in families and (method,low) in families:\n                    pair(families[(method,high)],families[(method,low)],f\'{method} alpha={high:g} minus alpha={low:g}\',\'alpha_effect\')\n    ds=[]\n    for label in dict.fromkeys(r[\'comparison\'] for r in deltas):\n        rr=[r for r in deltas if r[\'comparison\']==label]\n        ds.append(dict(category=rr[0][\'category\'],comparison=label,n_seeds=3,\n            **{k+\'_mean\':float(np.mean([r[k] for r in rr])) for k in METRICS},\n            **{k+\'_seed_sd\':float(np.std([r[k] for r in rr],ddof=1)) for k in METRICS}))\n    table(out/\'all_18_conditions.csv\',inv)\n    table(out/\'combined_Day2_KD_per_seed.csv\',allrows)\n    table(out/\'combined_Day2_KD_summary.csv\',summary)\n    table(out/\'paired_deltas.csv\',deltas,fields=[\'category\',\'comparison\',\'seed\']+METRICS)\n    table(out/\'paired_delta_summary.csv\',ds,fields=[\'category\',\'comparison\',\'n_seeds\']+[k+s for s in [\'_mean\',\'_seed_sd\'] for k in METRICS])\n    legacy_csv=ctx[\'work\']/\'legacy_lr0001/reports/val_select/per_seed.csv\'\n    legacy_rows=[]\n    if legacy_csv.exists():\n        for r in readcsv(legacy_csv):\n            if r.get(\'method\',\'\').startswith(\'KD\'):\n                legacy_rows.append(dict(r,source=\'legacy_supplement\',lr=.001,counted_in_18=False,comparison_scope=\'historical supplementary only\'))\n    table(out/\'legacy_KD_lr0001_supplement.csv\',legacy_rows)\n    pending=[r for r in inv if r[\'status\']!=\'complete\']\n    status=dict(expected_conditions=18,completed_conditions=18-len(pending),complete_three_seed_groups=len(families),complete=len(pending)==0,\n        missing_or_incomplete=pending,legacy_status=ctx[\'protocol\'][\'legacy_status\'],teacher_params=ctx[\'params\'],\n        scope=\'val_select development only; checkpoint-selected; no test evaluation; no single-seed conclusions\',\n        student_postcalibration=\'Day2 results preserved; no new postcalibration or refitting\')\n    e.save(out/\'status.json\',status)\n    text=f\'# Stage2 Hazel\\n\\n完成 {18-len(pending)}/18 个条件、{len(families)}/6 个完整三种子配置。\\n\\n\'\n    text+=\'只对完整三种子配置汇总和配对；未完成种子仅保留记录，不下结论。原始学生 logits 用于指标，蒸馏损失内部使用 s/tau 与 tau²。教师参数仅来自本轮交付，不重复拟合。\\n\\n\'\n    text+=\'combined 表接续 Day2 公共学习率下 CE、LS 与学生后校准；Day2_preserved 保留全部学习率比较与模型。旧 lr=.001 KD 如提供，位于 legacy_lr0001，不计入18条件。\\n\\n\'\n    text+=\'Macro-F1、Accuracy、各类F1、ECE 为0–1原值，差值乘100为百分点。NLL为自然对数，Brier为三类平方误差之和；NLL/ECE/Brier越小越好。标准差为种子间样本标准差，不是置信区间。开发集已用于选模及历史决策，不是独立测试。\\n\'\n    e.atomic_bytes(out/\'README.md\',text.encode(\'utf-8\'))\n    return out\n\ndef package(ctx):\n    report(ctx)\n    out=ctx[\'work\']/(\'Stage2_Hazel_results_\'+uuid.uuid4().hex[:8]+\'.zip\')\n    files=[ctx[\'work\']/\'protocol.json\']\n    for folder in [\'code\',\'teacher_receipt\',\'Day2_preserved\',\'legacy_lr0001\',\'groups\',\'report\']:\n        files += [p for p in (ctx[\'work\']/folder).rglob(\'*\') if p.is_file() and p.suffix in [\'.json\',\'.csv\',\'.py\',\'.pt\',\'.md\'] and p.name!=\'state.pt\']\n    hashes={p.relative_to(ctx[\'work\']).as_posix():e.sha(p) for p in files}\n    with zipfile.ZipFile(out,\'x\',compression=zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(set(files)): z.write(p,p.relative_to(ctx[\'work\']).as_posix())\n        z.writestr(\'PACKAGE_SHA256.json\',json.dumps(hashes,indent=2))\n    return out\n'}
CODE_HASHES={'cache_audit.py': 'fc464669c08fd73dfd0449a92ba3b37e28595debc64aa32f0b11eac559cebd65', 'experiment.py': 'c1ee5ba6c4aae85d8ac64cbc6c06d8ee3f0b673bf14d0b31f35133cddcfbbe86', 'stage2.py': 'd0ed64fa7a9842abed927c17e6df258420277b4f9b6f67f6225f545f8e39aa8e'}
TEACHER_BYTES={'teacher_calibration_stage2.json': 'ewogICJzY2hlbWEiOiAic3RhZ2UyX3RlYWNoZXJfZGV2Y2FsX3YxIiwKICAibGFiZWxzIjogWwogICAgIkUiLAogICAgIk4iLAogICAgIkMiCiAgXSwKICAiZml0X3NwbGl0IjogInZhbF9maXQiLAogICJmaXRfbiI6IDQwMCwKICAiZm9ybXVsYSI6ICIocmF3X2xvZ2l0cyArIFtiLDAsMF0pIC8gVCIsCiAgInByb3RvY29sX3NoYTI1NiI6ICJkZTkyNmYzOTg4MmNjM2U2MzYwYTkxM2E4ODdiNDJmODk0Mzg4OTAwMDMxOTMwMTA3OWMxN2M5NzAxNDFiN2FjIiwKICAiaW5wdXRfc2hhMjU2IjogewogICAgInZhbF9maXQuanNvbmwiOiAiYTU4ZTJmMDAwNDdmMTU5Y2E3YjY0MTU1NjRmZjAzYjY5OWM2ODg5N2ZlYTgwNWZiZmI5MDg4NzI5ZmRiYmFmNCIsCiAgICAiYmFzZS5qc29ubCI6ICI0YzYxMDIzNTVlNzM4MmVmNDkzNmZlNjljOTllMDRlZTY2NTMxOWUwMWE4MTQzNGU2MmJmZGViYmE0MGVjZjU3IiwKICAgICJqZXYuanNvbmwiOiAiNmY4NWIyMGYzY2MwY2MzNzAxYmYwZWU2NDliYzdmNmJiYzgyYjU5OGE1MjQyMDlhNWQ3ODQyNDZjZmFiZDQ2YSIsCiAgICAidGVhY2hlcl9ydW4uanNvbiI6ICJhNDI1YTFjYzI4MTFlYmJjODVlMTAzM2JhODRkYWU4YTA2MzJjY2Q4ZTUzYjhkN2M1YTRmMmZjZTE5MzU5MjliIiwKICAgICJpbWFnZV9oYXNoX3Byb2plY3Rpb24uanNvbiI6ICJkNGYwMGEyOGMwZTE3NDBhYTY3ZmZiNjRhNjM2MmFmNWYxNDc5NTQ1MmFhYzBmMjA5OTE0ZTk0ODVlOWFiOGQwIiwKICAgICJwcm92ZW5hbmNlLmpzb24iOiAiYTQwOWZjOGMwZjUyN2Q5MDZiNzBiNmExNDdhODIwNWViYzI5MjNmZjAzMTFlNzNhM2RhNmI4OGM3MjlmMGQ4MiIKICB9LAogICJleHBlY3RlZF90cmFpbmluZ19pbnB1dHMiOiB7CiAgICAidHJhaW5fbWFuaWZlc3QiOiB7CiAgICAgICJzaGEyNTYiOiAiOWRjNDQwODgwY2RhMTdlMTMzMjYzMGI3MDMyNjMwNjhiNDY4OTEyMDM0ZTE2NGEwN2U0MWMwNzRkMWQ1NmY4NSIsCiAgICAgICJyZWxhdGl2ZV9wYXRoIjogInJldXNlZF92MS9oYW5kb2ZmL21hbmlmZXN0cy90cmFpbi5qc29ubCIKICAgIH0sCiAgICAiYmFzZV90cmFpbiI6IHsKICAgICAgInNoYTI1NiI6ICI5ZTcyZDNkN2Q1NDQwNzJjYzY4NDVlNDNiMWFjYTI2YTI4MzI1MGMxM2JhNGVhMTMzODJmYmFkYmIwMTIxNjU0IiwKICAgICAgInJlbGF0aXZlX3BhdGgiOiAicmV1c2VkX3YxL2hhbmRvZmYvcHJlZGljdGlvbnMvdHJhaW4vYmFzZS5qc29ubCIKICAgIH0sCiAgICAiamV2X3RyYWluIjogewogICAgICAic2hhMjU2IjogImIxZTVhNWFhMDJkZTk3ZDQ3ODcwYTRkMzY5NWQ0ZjA2NTFmNWU4NWJlMzVlZGE2YjQ5N2YzZTYyNjE5YmEzOTUiLAogICAgICAicmVsYXRpdmVfcGF0aCI6ICJyZXVzZWRfdjEvaGFuZG9mZi9wcmVkaWN0aW9ucy90cmFpbi9qZXYuanNvbmwiCiAgICB9CiAgfSwKICAidGVhY2hlcl9yZXZpc2lvbnMiOiB7CiAgICAiYmFzZV9zaGEiOiAiZWJiMjgxZWM3MGIwNTA5MGFhNjE2NWIwMTZlYWM4ZWMwOGU3MWIxNyIsCiAgICAiYWRhcHRlcl9zaGEiOiAiN2EzZjFiYjBkOGY3YmZiYmQ4NTE0ZDk2OGFiN2RlMzUxODg0OWM4ZCIsCiAgICAiZGF0YXNldF9zaGEiOiAiNTRlMDdlMzcwZmZiMTljMWI1ZWViZjEzMDU1ZThiNmFmMzM5OTFhZCIsCiAgICAiamV2X2NvbW1pdCI6ICJhMzkyZWFlZGZhNDFhZGFlMzQ4Zjk3NjdiNTVjMGY1NGIzYTEzYjU3IgogIH0sCiAgInRlYWNoZXJfZHR5cGUiOiAiZmxvYXQxNiIsCiAgIm5vdGUiOiAiRGV2LUNhbCBmaXQgZGlhZ25vc3RpY3Mgb25seTsgYm90aCBiIGFuZCBUIGZpdHRlZCBvbiB2YWxfZml0LiBGaXhlZCBmdWxsIGNhbGlicmF0aW9uLCBub3QgdmFyaWFudCBzZWxlY3Rpb24gb24gdmFsX3NlbGVjdC4iLAogICJtb2RlbHMiOiB7CiAgICAiYmFzZSI6IHsKICAgICAgInZhcmlhbnRzIjogewogICAgICAgICJyYXciOiB7CiAgICAgICAgICAiYiI6IDAuMCwKICAgICAgICAgICJUIjogMS4wCiAgICAgICAgfSwKICAgICAgICAidGVtcGVyYXR1cmVfY29tcG9uZW50IjogewogICAgICAgICAgImIiOiAwLjAsCiAgICAgICAgICAiVCI6IDQuMAogICAgICAgIH0sCiAgICAgICAgIkVfYmlhcyI6IHsKICAgICAgICAgICJiIjogLTIuMCwKICAgICAgICAgICJUIjogMS4wCiAgICAgICAgfSwKICAgICAgICAiRV9iaWFzX3RlbXBlcmF0dXJlIjogewogICAgICAgICAgImIiOiAtMi4wLAogICAgICAgICAgIlQiOiA0LjAKICAgICAgICB9CiAgICAgIH0sCiAgICAgICJzZWxlY3RlZCI6ICJFX2JpYXNfdGVtcGVyYXR1cmUiLAogICAgICAic2VsZWN0ZWRfcGFyYW1zIjogewogICAgICAgICJiIjogLTIuMCwKICAgICAgICAiVCI6IDQuMAogICAgICB9LAogICAgICAic2VsZWN0aW9uIjogImZpeGVkX2J5X3Byb3RvY29sX25vdF9ieV92YWxpZGF0aW9uX3ZhcmlhbnRfc2NvcmVzIiwKICAgICAgImJpYXNfYXRfYm91bmRhcnkiOiB0cnVlLAogICAgICAidGVtcGVyYXR1cmVfYXRfYm91bmRhcnkiOiB0cnVlLAogICAgICAiZml0X2RpYWdub3N0aWNzIjogewogICAgICAgICJyYXciOiB7CiAgICAgICAgICAibiI6IDQwMCwKICAgICAgICAgICJhY2N1cmFjeSI6IDAuNTksCiAgICAgICAgICAibWFjcm9fZjEiOiAwLjU5MTAyNjc5MTc4NTY2NTgsCiAgICAgICAgICAiZjFfRSI6IDAuNjg2NjY2NjY2NjY2NjY2NiwKICAgICAgICAgICJmMV9OIjogMC40NjMxNTc4OTQ3MzY4NDIxLAogICAgICAgICAgImYxX0MiOiAwLjYyMzI1NTgxMzk1MzQ4ODQsCiAgICAgICAgICAibmxsIjogMy4xMzc4MDgzMDQ3MDI1NDczLAogICAgICAgICAgImVjZV8xMCI6IDAuMzUzMzk3NDc2MTgyMzEwMjcsCiAgICAgICAgICAiYnJpZXIiOiAwLjc1MTgxMDM2NDQ4MTUyMDksCiAgICAgICAgICAiY29uZnVzaW9uX21hdHJpeCI6IFsKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDEwMywKICAgICAgICAgICAgICA1LAogICAgICAgICAgICAgIDEKICAgICAgICAgICAgXSwKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDc4LAogICAgICAgICAgICAgIDY2LAogICAgICAgICAgICAgIDgKICAgICAgICAgICAgXSwKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDEwLAogICAgICAgICAgICAgIDYyLAogICAgICAgICAgICAgIDY3CiAgICAgICAgICAgIF0KICAgICAgICAgIF0KICAgICAgICB9LAogICAgICAgICJ0ZW1wZXJhdHVyZV9jb21wb25lbnQiOiB7CiAgICAgICAgICAibiI6IDQwMCwKICAgICAgICAgICJhY2N1cmFjeSI6IDAuNTksCiAgICAgICAgICAibWFjcm9fZjEiOiAwLjU5MTAyNjc5MTc4NTY2NTgsCiAgICAgICAgICAiZjFfRSI6IDAuNjg2NjY2NjY2NjY2NjY2NiwKICAgICAgICAgICJmMV9OIjogMC40NjMxNTc4OTQ3MzY4NDIxLAogICAgICAgICAgImYxX0MiOiAwLjYyMzI1NTgxMzk1MzQ4ODQsCiAgICAgICAgICAibmxsIjogMS4wNDQ2MDYwNTAzMDE0MzI1LAogICAgICAgICAgImVjZV8xMCI6IDAuMTkwNzU2Mjc0NTU5ODA3MzcsCiAgICAgICAgICAiYnJpZXIiOiAwLjYwNzYzNDgwNTg4MTQ5MDgsCiAgICAgICAgICAiY29uZnVzaW9uX21hdHJpeCI6IFsKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDEwMywKICAgICAgICAgICAgICA1LAogICAgICAgICAgICAgIDEKICAgICAgICAgICAgXSwKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDc4LAogICAgICAgICAgICAgIDY2LAogICAgICAgICAgICAgIDgKICAgICAgICAgICAgXSwKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDEwLAogICAgICAgICAgICAgIDYyLAogICAgICAgICAgICAgIDY3CiAgICAgICAgICAgIF0KICAgICAgICAgIF0KICAgICAgICB9LAogICAgICAgICJFX2JpYXMiOiB7CiAgICAgICAgICAibiI6IDQwMCwKICAgICAgICAgICJhY2N1cmFjeSI6IDAuNjAyNSwKICAgICAgICAgICJtYWNyb19mMSI6IDAuNjA1MTUzNjY0OTcwOTk4MiwKICAgICAgICAgICJmMV9FIjogMC42OTg5NjE5Mzc3MTYyNjMsCiAgICAgICAgICAiZjFfTiI6IDAuNDkzMjQzMjQzMjQzMjQzMjYsCiAgICAgICAgICAiZjFfQyI6IDAuNjIzMjU1ODEzOTUzNDg4NCwKICAgICAgICAgICJubGwiOiAyLjc1MjM2MTcwMjI0NDcxNywKICAgICAgICAgICJlY2VfMTAiOiAwLjM0OTU2MTgzNDg2NDI0ODMsCiAgICAgICAgICAiYnJpZXIiOiAwLjcyNzQ3NDE3MzMwMjcwMTcsCiAgICAgICAgICAiY29uZnVzaW9uX21hdHJpeCI6IFsKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDEwMSwKICAgICAgICAgICAgICA3LAogICAgICAgICAgICAgIDEKICAgICAgICAgICAgXSwKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDcxLAogICAgICAgICAgICAgIDczLAogICAgICAgICAgICAgIDgKICAgICAgICAgICAgXSwKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDgsCiAgICAgICAgICAgICAgNjQsCiAgICAgICAgICAgICAgNjcKICAgICAgICAgICAgXQogICAgICAgICAgXQogICAgICAgIH0sCiAgICAgICAgIkVfYmlhc190ZW1wZXJhdHVyZSI6IHsKICAgICAgICAgICJuIjogNDAwLAogICAgICAgICAgImFjY3VyYWN5IjogMC42MDI1LAogICAgICAgICAgIm1hY3JvX2YxIjogMC42MDUxNTM2NjQ5NzA5OTgyLAogICAgICAgICAgImYxX0UiOiAwLjY5ODk2MTkzNzcxNjI2MywKICAgICAgICAgICJmMV9OIjogMC40OTMyNDMyNDMyNDMyNDMyNiwKICAgICAgICAgICJmMV9DIjogMC42MjMyNTU4MTM5NTM0ODg0LAogICAgICAgICAgIm5sbCI6IDAuOTU1MDQzMjIwMzg3ODg2MSwKICAgICAgICAgICJlY2VfMTAiOiAwLjE3MTUxMTk5MzU1ODY3NDI3LAogICAgICAgICAgImJyaWVyIjogMC41NzM1OTYxMzYwNDU3MjU1LAogICAgICAgICAgImNvbmZ1c2lvbl9tYXRyaXgiOiBbCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICAxMDEsCiAgICAgICAgICAgICAgNywKICAgICAgICAgICAgICAxCiAgICAgICAgICAgIF0sCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICA3MSwKICAgICAgICAgICAgICA3MywKICAgICAgICAgICAgICA4CiAgICAgICAgICAgIF0sCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICA4LAogICAgICAgICAgICAgIDY0LAogICAgICAgICAgICAgIDY3CiAgICAgICAgICAgIF0KICAgICAgICAgIF0KICAgICAgICB9CiAgICAgIH0KICAgIH0sCiAgICAiamV2IjogewogICAgICAidmFyaWFudHMiOiB7CiAgICAgICAgInJhdyI6IHsKICAgICAgICAgICJiIjogMC4wLAogICAgICAgICAgIlQiOiAxLjAKICAgICAgICB9LAogICAgICAgICJ0ZW1wZXJhdHVyZV9jb21wb25lbnQiOiB7CiAgICAgICAgICAiYiI6IDAuMCwKICAgICAgICAgICJUIjogMS4zNDI1NzI1MDI3ODAyNjM1CiAgICAgICAgfSwKICAgICAgICAiRV9iaWFzIjogewogICAgICAgICAgImIiOiAwLjE0OTk5OTk5OTk5OTk5OTksCiAgICAgICAgICAiVCI6IDEuMAogICAgICAgIH0sCiAgICAgICAgIkVfYmlhc190ZW1wZXJhdHVyZSI6IHsKICAgICAgICAgICJiIjogMC4xNDk5OTk5OTk5OTk5OTk5LAogICAgICAgICAgIlQiOiAxLjM0MjU3MjUwMjc4MDI2MzUKICAgICAgICB9CiAgICAgIH0sCiAgICAgICJzZWxlY3RlZCI6ICJFX2JpYXNfdGVtcGVyYXR1cmUiLAogICAgICAic2VsZWN0ZWRfcGFyYW1zIjogewogICAgICAgICJiIjogMC4xNDk5OTk5OTk5OTk5OTk5LAogICAgICAgICJUIjogMS4zNDI1NzI1MDI3ODAyNjM1CiAgICAgIH0sCiAgICAgICJzZWxlY3Rpb24iOiAiZml4ZWRfYnlfcHJvdG9jb2xfbm90X2J5X3ZhbGlkYXRpb25fdmFyaWFudF9zY29yZXMiLAogICAgICAiYmlhc19hdF9ib3VuZGFyeSI6IGZhbHNlLAogICAgICAidGVtcGVyYXR1cmVfYXRfYm91bmRhcnkiOiBmYWxzZSwKICAgICAgImZpdF9kaWFnbm9zdGljcyI6IHsKICAgICAgICAicmF3IjogewogICAgICAgICAgIm4iOiA0MDAsCiAgICAgICAgICAiYWNjdXJhY3kiOiAwLjc5LAogICAgICAgICAgIm1hY3JvX2YxIjogMC43OTAyMTc2NzQ1NjY3NTAyLAogICAgICAgICAgImYxX0UiOiAwLjc2OTg3NDQ3Njk4NzQ0NzcsCiAgICAgICAgICAiZjFfTiI6IDAuNzQwNDg0NDI5MDY1NzQ0LAogICAgICAgICAgImYxX0MiOiAwLjg2MDI5NDExNzY0NzA1ODksCiAgICAgICAgICAibmxsIjogMC41NDYwNzA4OTgzOTI3OTc5LAogICAgICAgICAgImVjZV8xMCI6IDAuMDgyNTE1NjE4OTM2Nzg0MjUsCiAgICAgICAgICAiYnJpZXIiOiAwLjMxMzEzMDMzMTg5NTY1NDE1LAogICAgICAgICAgImNvbmZ1c2lvbl9tYXRyaXgiOiBbCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICA5MiwKICAgICAgICAgICAgICAxNSwKICAgICAgICAgICAgICAyCiAgICAgICAgICAgIF0sCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICAzMSwKICAgICAgICAgICAgICAxMDcsCiAgICAgICAgICAgICAgMTQKICAgICAgICAgICAgXSwKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDcsCiAgICAgICAgICAgICAgMTUsCiAgICAgICAgICAgICAgMTE3CiAgICAgICAgICAgIF0KICAgICAgICAgIF0KICAgICAgICB9LAogICAgICAgICJ0ZW1wZXJhdHVyZV9jb21wb25lbnQiOiB7CiAgICAgICAgICAibiI6IDQwMCwKICAgICAgICAgICJhY2N1cmFjeSI6IDAuNzksCiAgICAgICAgICAibWFjcm9fZjEiOiAwLjc5MDIxNzY3NDU2Njc1MDIsCiAgICAgICAgICAiZjFfRSI6IDAuNzY5ODc0NDc2OTg3NDQ3NywKICAgICAgICAgICJmMV9OIjogMC43NDA0ODQ0MjkwNjU3NDQsCiAgICAgICAgICAiZjFfQyI6IDAuODYwMjk0MTE3NjQ3MDU4OSwKICAgICAgICAgICJubGwiOiAwLjUyMzU3MDgxMDg2MTYzMzYsCiAgICAgICAgICAiZWNlXzEwIjogMC4wMjMyNjE0MTQxNjUzNjIyNDYsCiAgICAgICAgICAiYnJpZXIiOiAwLjMwMjQ2MTU2MDAwNjI5NjI2LAogICAgICAgICAgImNvbmZ1c2lvbl9tYXRyaXgiOiBbCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICA5MiwKICAgICAgICAgICAgICAxNSwKICAgICAgICAgICAgICAyCiAgICAgICAgICAgIF0sCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICAzMSwKICAgICAgICAgICAgICAxMDcsCiAgICAgICAgICAgICAgMTQKICAgICAgICAgICAgXSwKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDcsCiAgICAgICAgICAgICAgMTUsCiAgICAgICAgICAgICAgMTE3CiAgICAgICAgICAgIF0KICAgICAgICAgIF0KICAgICAgICB9LAogICAgICAgICJFX2JpYXMiOiB7CiAgICAgICAgICAibiI6IDQwMCwKICAgICAgICAgICJhY2N1cmFjeSI6IDAuNzksCiAgICAgICAgICAibWFjcm9fZjEiOiAwLjc5MDUzMjU4OTYyOTI0NzMsCiAgICAgICAgICAiZjFfRSI6IDAuNzczNjYyNTUxNDQwMzI5MiwKICAgICAgICAgICJmMV9OIjogMC43Mzg2NzU5NTgxODgxNTMzLAogICAgICAgICAgImYxX0MiOiAwLjg1OTI1OTI1OTI1OTI1OTMsCiAgICAgICAgICAibmxsIjogMC41NTIwOTUzMTQ5MzgzNTA2LAogICAgICAgICAgImVjZV8xMCI6IDAuMDc5OTU3MjUzMjQxMzc4NjksCiAgICAgICAgICAiYnJpZXIiOiAwLjMxNTM5NDgxNTUwMDc2MzgsCiAgICAgICAgICAiY29uZnVzaW9uX21hdHJpeCI6IFsKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDk0LAogICAgICAgICAgICAgIDE0LAogICAgICAgICAgICAgIDEKICAgICAgICAgICAgXSwKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDMyLAogICAgICAgICAgICAgIDEwNiwKICAgICAgICAgICAgICAxNAogICAgICAgICAgICBdLAogICAgICAgICAgICBbCiAgICAgICAgICAgICAgOCwKICAgICAgICAgICAgICAxNSwKICAgICAgICAgICAgICAxMTYKICAgICAgICAgICAgXQogICAgICAgICAgXQogICAgICAgIH0sCiAgICAgICAgIkVfYmlhc190ZW1wZXJhdHVyZSI6IHsKICAgICAgICAgICJuIjogNDAwLAogICAgICAgICAgImFjY3VyYWN5IjogMC43OSwKICAgICAgICAgICJtYWNyb19mMSI6IDAuNzkwNTMyNTg5NjI5MjQ3MywKICAgICAgICAgICJmMV9FIjogMC43NzM2NjI1NTE0NDAzMjkyLAogICAgICAgICAgImYxX04iOiAwLjczODY3NTk1ODE4ODE1MzMsCiAgICAgICAgICAiZjFfQyI6IDAuODU5MjU5MjU5MjU5MjU5MywKICAgICAgICAgICJubGwiOiAwLjUyNzgwNDAxMjc2MDE5NDgsCiAgICAgICAgICAiZWNlXzEwIjogMC4wMjg4NTY1NTUxMjkyNjYzMzYsCiAgICAgICAgICAiYnJpZXIiOiAwLjMwNDIyNTE0ODM4Njg0MzQ1LAogICAgICAgICAgImNvbmZ1c2lvbl9tYXRyaXgiOiBbCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICA5NCwKICAgICAgICAgICAgICAxNCwKICAgICAgICAgICAgICAxCiAgICAgICAgICAgIF0sCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICAzMiwKICAgICAgICAgICAgICAxMDYsCiAgICAgICAgICAgICAgMTQKICAgICAgICAgICAgXSwKICAgICAgICAgICAgWwogICAgICAgICAgICAgIDgsCiAgICAgICAgICAgICAgMTUsCiAgICAgICAgICAgICAgMTE2CiAgICAgICAgICAgIF0KICAgICAgICAgIF0KICAgICAgICB9CiAgICAgIH0KICAgIH0KICB9Cn0=', 'protocol_LOCKED.json': 'ewogICJzY2hlbWEiOiAic3RhZ2UyX3RlYWNoZXJfZGV2Y2FsX3YxIiwKICAibGFiZWxfb3JkZXIiOiBbCiAgICAiRSIsCiAgICAiTiIsCiAgICAiQyIKICBdLAogICJmaXRfc3BsaXQiOiAidmFsX2ZpdCIsCiAgImZpdF9uIjogNDAwLAogICJiaWFzX2dyaWQiOiBbCiAgICAtMi4wLAogICAgLTEuOTUsCiAgICAtMS45LAogICAgLTEuODUsCiAgICAtMS44LAogICAgLTEuNzUsCiAgICAtMS43LAogICAgLTEuNjUsCiAgICAtMS42LAogICAgLTEuNTUsCiAgICAtMS41LAogICAgLTEuNDUsCiAgICAtMS40LAogICAgLTEuMzUsCiAgICAtMS4yOTk5OTk5OTk5OTk5OTk4LAogICAgLTEuMjUsCiAgICAtMS4yLAogICAgLTEuMTUsCiAgICAtMS4xLAogICAgLTEuMDQ5OTk5OTk5OTk5OTk5OCwKICAgIC0xLjAsCiAgICAtMC45NSwKICAgIC0wLjg5OTk5OTk5OTk5OTk5OTksCiAgICAtMC44NDk5OTk5OTk5OTk5OTk5LAogICAgLTAuNzk5OTk5OTk5OTk5OTk5OCwKICAgIC0wLjc1LAogICAgLTAuNywKICAgIC0wLjY0OTk5OTk5OTk5OTk5OTksCiAgICAtMC41OTk5OTk5OTk5OTk5OTk5LAogICAgLTAuNTQ5OTk5OTk5OTk5OTk5OCwKICAgIC0wLjUsCiAgICAtMC40NDk5OTk5OTk5OTk5OTk5NiwKICAgIC0wLjM5OTk5OTk5OTk5OTk5OTksCiAgICAtMC4zNDk5OTk5OTk5OTk5OTk4NywKICAgIC0wLjI5OTk5OTk5OTk5OTk5OTgsCiAgICAtMC4yNSwKICAgIC0wLjE5OTk5OTk5OTk5OTk5OTk2LAogICAgLTAuMTQ5OTk5OTk5OTk5OTk5OSwKICAgIC0wLjA5OTk5OTk5OTk5OTk5OTg3LAogICAgLTAuMDQ5OTk5OTk5OTk5OTk5ODIsCiAgICAwLjAsCiAgICAwLjA1MDAwMDAwMDAwMDAwMDI2NiwKICAgIDAuMTAwMDAwMDAwMDAwMDAwMDksCiAgICAwLjE0OTk5OTk5OTk5OTk5OTksCiAgICAwLjIwMDAwMDAwMDAwMDAwMDE4LAogICAgMC4yNSwKICAgIDAuMzAwMDAwMDAwMDAwMDAwMjcsCiAgICAwLjM1MDAwMDAwMDAwMDAwMDEsCiAgICAwLjQwMDAwMDAwMDAwMDAwMDM2LAogICAgMC40NTAwMDAwMDAwMDAwMDAyLAogICAgMC41LAogICAgMC41NTAwMDAwMDAwMDAwMDAzLAogICAgMC42MDAwMDAwMDAwMDAwMDAxLAogICAgMC42NTAwMDAwMDAwMDAwMDA0LAogICAgMC43MDAwMDAwMDAwMDAwMDAyLAogICAgMC43NSwKICAgIDAuODAwMDAwMDAwMDAwMDAwMywKICAgIDAuODUwMDAwMDAwMDAwMDAwMSwKICAgIDAuOTAwMDAwMDAwMDAwMDAwNCwKICAgIDAuOTUwMDAwMDAwMDAwMDAwMiwKICAgIDEuMCwKICAgIDEuMDUwMDAwMDAwMDAwMDAwMywKICAgIDEuMSwKICAgIDEuMTUwMDAwMDAwMDAwMDAwNCwKICAgIDEuMjAwMDAwMDAwMDAwMDAwMiwKICAgIDEuMjUsCiAgICAxLjMwMDAwMDAwMDAwMDAwMDMsCiAgICAxLjM1LAogICAgMS40MDAwMDAwMDAwMDAwMDA0LAogICAgMS40NTAwMDAwMDAwMDAwMDAyLAogICAgMS41LAogICAgMS41NTAwMDAwMDAwMDAwMDAzLAogICAgMS42LAogICAgMS42NTAwMDAwMDAwMDAwMDA0LAogICAgMS43MDAwMDAwMDAwMDAwMDAyLAogICAgMS43NSwKICAgIDEuODAwMDAwMDAwMDAwMDAwMywKICAgIDEuODUsCiAgICAxLjkwMDAwMDAwMDAwMDAwMDQsCiAgICAxLjk1MDAwMDAwMDAwMDAwMDIsCiAgICAyLjAKICBdLAogICJ0ZW1wZXJhdHVyZV9ncmlkIjogWwogICAgMC4yNSwKICAgIDAuMjU0MzY5OTIzMDI1NjcxNiwKICAgIDAuMjU4ODE2MjMwOTYwMzQ0NCwKICAgIDAuMjYzMzQwMjU4OTg4NzA4OTQsCiAgICAwLjI2Nzk0MzM2NTYzNDA3MzM0LAogICAgMC4yNzI2MjY5MzMxNjYzMTQ0LAogICAgMC4yNzczOTIzNjgwMTY5NjEyLAogICAgMC4yODIyNDExMDEyMDE1MzI4NCwKICAgIDAuMjg3MTc0NTg4NzQ5MjU4NywKICAgIDAuMjkyMTk0MzEyMTQwMzExMzMsCiAgICAwLjI5NzMwMTc3ODc1MDY4MDI2LAogICAgMC4zMDI0OTg1MjIzMDQ4MjMxLAogICAgMC4zMDc3ODYxMDMzMzYyMjkxLAogICAgMC4zMTMxNjYxMDk2NTYwMzIsCiAgICAwLjMxODY0MDE1NjgyOTgxNTU2LAogICAgMC4zMjQyMDk4ODg2NjI3NTI0LAogICAgMC4zMjk4NzY5Nzc2OTMyMjM1NSwKICAgIDAuMzM1NjQzMTI1Njk1MDY1OSwKICAgIDAuMzQxNTEwMDY0MTg4NTk4ODYsCiAgICAwLjM0NzQ3OTU1NDk2MDU4NDI1LAogICAgMC4zNTM1NTMzOTA1OTMyNzM4LAogICAgMC4zNTk3MzMzOTUwMDI3MDUsCiAgICAwLjM2NjAyMTQyMzk4NjQwNjM2LAogICAgMC4zNzI0MTkzNjU3ODA2NzU2LAogICAgMC4zNzg5MjkxNDE2Mjc1OTk1NSwKICAgIDAuMzg1NTUyNzA2MzUxOTg1MjQsCiAgICAwLjM5MjI5MjA0ODk0ODM3NTQsCiAgICAwLjM5OTE0OTE5MzE3ODMyNDksCiAgICAwLjQwNjEyNjE5ODE3ODExNzgsCiAgICAwLjQxMzIyNTE1OTA3NzEwNTgzLAogICAgMC40MjA0NDgyMDc2MjY4NTczLAogICAgMC40Mjc3OTc1MTI4NDEzMDExLAogICAgMC40MzUyNzUyODE2NDgwNjIwNiwKICAgIDAuNDQyODgzNzU5NTUxMTgwMywKICAgIDAuNDUwNjI1MjMxMzA1NDE1MTUsCiAgICAwLjQ1ODUwMjAyMTYwMjMzNTYsCiAgICAwLjQ2NjUxNjQ5NTc2ODQwMzcsCiAgICAwLjQ3NDY3MTA2MDQ3NTI1OTY0LAogICAgMC40ODI5NjgxNjQ0NjI0MjI4LAogICAgMC40OTE0MTAyOTkyNzI2MjU1MywKICAgIDAuNSwKICAgIDAuNTA4NzM5ODQ2MDUxMzQzMywKICAgIDAuNTE3NjMyNDYxOTIwNjg4OCwKICAgIDAuNTI2NjgwNTE3OTc3NDE4LAogICAgMC41MzU4ODY3MzEyNjgxNDY2LAogICAgMC41NDUyNTM4NjYzMzI2Mjg4LAogICAgMC41NTQ3ODQ3MzYwMzM5MjI1LAogICAgMC41NjQ0ODIyMDI0MDMwNjU3LAogICAgMC41NzQzNDkxNzc0OTg1MTc1LAogICAgMC41ODQzODg2MjQyODA2MjI4LAogICAgMC41OTQ2MDM1NTc1MDEzNjA1LAogICAgMC42MDQ5OTcwNDQ2MDk2NDYzLAogICAgMC42MTU1NzIyMDY2NzI0NTgyLAogICAgMC42MjYzMzIyMTkzMTIwNjQsCiAgICAwLjYzNzI4MDMxMzY1OTYzMTEsCiAgICAwLjY0ODQxOTc3NzMyNTUwNDgsCiAgICAwLjY1OTc1Mzk1NTM4NjQ0NzIsCiAgICAwLjY3MTI4NjI1MTM5MDEzMTgsCiAgICAwLjY4MzAyMDEyODM3NzE5NzcsCiAgICAwLjY5NDk1OTEwOTkyMTE2ODUsCiAgICAwLjcwNzEwNjc4MTE4NjU0NzYsCiAgICAwLjcxOTQ2Njc5MDAwNTQxLAogICAgMC43MzIwNDI4NDc5NzI4MTI3LAogICAgMC43NDQ4Mzg3MzE1NjEzNTEyLAogICAgMC43NTc4NTgyODMyNTUxOTkxLAogICAgMC43NzExMDU0MTI3MDM5NzA1LAogICAgMC43ODQ1ODQwOTc4OTY3NTA4LAogICAgMC43OTgyOTgzODYzNTY2NSwKICAgIDAuODEyMjUyMzk2MzU2MjM1NywKICAgIDAuODI2NDUwMzE4MTU0MjExOCwKICAgIDAuODQwODk2NDE1MjUzNzE0NSwKICAgIDAuODU1NTk1MDI1NjgyNjAyMywKICAgIDAuODcwNTUwNTYzMjk2MTI0MSwKICAgIDAuODg1NzY3NTE5MTAyMzYwNywKICAgIDAuOTAxMjUwNDYyNjEwODMwMywKICAgIDAuOTE3MDA0MDQzMjA0NjcxMiwKICAgIDAuOTMzMDMyOTkxNTM2ODA3NSwKICAgIDAuOTQ5MzQyMTIwOTUwNTE5MywKICAgIDAuOTY1OTM2MzI4OTI0ODQ1NSwKICAgIDAuOTgyODIwNTk4NTQ1MjUxMiwKICAgIDEuMCwKICAgIDEuMDE3NDc5NjkyMTAyNjg2NSwKICAgIDEuMDM1MjY0OTIzODQxMzc3NiwKICAgIDEuMDUzMzYxMDM1OTU0ODM1OCwKICAgIDEuMDcxNzczNDYyNTM2MjkzNCwKICAgIDEuMDkwNTA3NzMyNjY1MjU3NywKICAgIDEuMTA5NTY5NDcyMDY3ODQ1MywKICAgIDEuMTI4OTY0NDA0ODA2MTMxNCwKICAgIDEuMTQ4Njk4MzU0OTk3MDM0OSwKICAgIDEuMTY4Nzc3MjQ4NTYxMjQ1OCwKICAgIDEuMTg5MjA3MTE1MDAyNzIxLAogICAgMS4yMDk5OTQwODkyMTkyOTI5LAogICAgMS4yMzExNDQ0MTMzNDQ5MTYzLAogICAgMS4yNTI2NjQ0Mzg2MjQxMjgsCiAgICAxLjI3NDU2MDYyNzMxOTI2MjIsCiAgICAxLjI5NjgzOTU1NDY1MTAwOTYsCiAgICAxLjMxOTUwNzkxMDc3Mjg5NDYsCiAgICAxLjM0MjU3MjUwMjc4MDI2MzUsCiAgICAxLjM2NjA0MDI1Njc1NDM5NTQsCiAgICAxLjM4OTkxODIxOTg0MjMzNzIsCiAgICAxLjQxNDIxMzU2MjM3MzA5NTEsCiAgICAxLjQzODkzMzU4MDAxMDgyLAogICAgMS40NjQwODU2OTU5NDU2MjU3LAogICAgMS40ODk2Nzc0NjMxMjI3MDIzLAogICAgMS41MTU3MTY1NjY1MTAzOTg0LAogICAgMS41NDIyMTA4MjU0MDc5NDEsCiAgICAxLjU2OTE2ODE5NTc5MzUwMTUsCiAgICAxLjU5NjU5Njc3MjcxMzMsCiAgICAxLjYyNDUwNDc5MjcxMjQ3MTIsCiAgICAxLjY1MjkwMDYzNjMwODQyMzgsCiAgICAxLjY4MTc5MjgzMDUwNzQyOTIsCiAgICAxLjcxMTE5MDA1MTM2NTIwNDMsCiAgICAxLjc0MTEwMTEyNjU5MjI0ODUsCiAgICAxLjc3MTUzNTAzODIwNDcyMTIsCiAgICAxLjgwMjUwMDkyNTIyMTY2MDgsCiAgICAxLjgzNDAwODA4NjQwOTM0MjcsCiAgICAxLjg2NjA2NTk4MzA3MzYxNDgsCiAgICAxLjg5ODY4NDI0MTkwMTAzODgsCiAgICAxLjkzMTg3MjY1Nzg0OTY5MTIsCiAgICAxLjk2NTY0MTE5NzA5MDUwMjEsCiAgICAyLjAwMDAwMDAwMDAwMDAwMDQsCiAgICAyLjAzNDk1OTM4NDIwNTM3MywKICAgIDIuMDcwNTI5ODQ3NjgyNzU1NiwKICAgIDIuMTA2NzIyMDcxOTA5NjcyLAogICAgMi4xNDM1NDY5MjUwNzI1ODYzLAogICAgMi4xODEwMTU0NjUzMzA1MTYsCiAgICAyLjIxOTEzODk0NDEzNTY5LAogICAgMi4yNTc5Mjg4MDk2MTIyNjI3LAogICAgMi4yOTczOTY3MDk5OTQwNywKICAgIDIuMzM3NTU0NDk3MTIyNDkxLAogICAgMi4zNzg0MTQyMzAwMDU0NDI1LAogICAgMi40MTk5ODgxNzg0Mzg1ODUzLAogICAgMi40NjIyODg4MjY2ODk4MzMsCiAgICAyLjUwNTMyODg3NzI0ODI1NjMsCiAgICAyLjU0OTEyMTI1NDYzODUyNDUsCiAgICAyLjU5MzY3OTEwOTMwMjAxOTMsCiAgICAyLjYzOTAxNTgyMTU0NTc4OTMsCiAgICAyLjY4NTE0NTAwNTU2MDUyNzUsCiAgICAyLjczMjA4MDUxMzUwODc5MTMsCiAgICAyLjc3OTgzNjQzOTY4NDY3NCwKICAgIDIuODI4NDI3MTI0NzQ2MTksCiAgICAyLjg3Nzg2NzE2MDAyMTY0MSwKICAgIDIuOTI4MTcxMzkxODkxMjUxOCwKICAgIDIuOTc5MzU0OTI2MjQ1NDA1LAogICAgMy4wMzE0MzMxMzMwMjA3OTY0LAogICAgMy4wODQ0MjE2NTA4MTU4ODE1LAogICAgMy4xMzgzMzYzOTE1ODcwMDQsCiAgICAzLjE5MzE5MzU0NTQyNjYwMDMsCiAgICAzLjI0OTAwOTU4NTQyNDk0MywKICAgIDMuMzA1ODAxMjcyNjE2ODQ3LAogICAgMy4zNjM1ODU2NjEwMTQ4NTgsCiAgICAzLjQyMjM4MDEwMjczMDQwOCwKICAgIDMuNDgyMjAyMjUzMTg0NDk3NCwKICAgIDMuNTQzMDcwMDc2NDA5NDQzLAogICAgMy42MDUwMDE4NTA0NDMzMjEsCiAgICAzLjY2ODAxNjE3MjgxODY4NSwKICAgIDMuNzMyMTMxOTY2MTQ3MjI5LAogICAgMy43OTczNjg0ODM4MDIwNzgsCiAgICAzLjg2Mzc0NTMxNTY5OTM4MywKICAgIDMuOTMxMjgyMzk0MTgxMDA0NywKICAgIDQuMAogIF0sCiAgImJpYXNfcnVsZSI6ICJUPTE7IG1heGltaXplIE1hY3JvLUYxLCB0aGVuIG1pbmltaXplIE5MTCwgdGhlbiBhYnMoYiksIHRoZW4gYjsgYWxsIG9uIHZhbF9maXQiLAogICJ0ZW1wZXJhdHVyZV9ydWxlIjogImFmdGVyIGZpeGluZyBiLCBtaW5pbWl6ZSBOTEwgb24gc2FtZSB2YWxfZml0OyB0aWVzIGFicyhsb2coVCkpLCB0aGVuIFQiLAogICJmb3JtdWxhIjogIihyYXdfbG9naXRzICsgW2IsMCwwXSkgLyBUIiwKICAidGVtcGVyYXR1cmVfY29tcG9uZW50IjogInVzZXMgdGhlIFQgZml0dGVkIEFGVEVSIGI7IG5vdCBzZXBhcmF0ZWx5IG9wdGltaXplZCBULW9ubHkiLAogICJwcmltYXJ5X3ZhcmlhbnQiOiAiRV9iaWFzX3RlbXBlcmF0dXJlIiwKICAidmFyaWFudF9zZWxlY3Rpb25fcGVyZm9ybWVkIjogZmFsc2UsCiAgImtkX3RhdSI6IDIuMCwKICAia2RfZm9ybXVsYSI6ICJzb2Z0bWF4KCgocmF3X2xvZ2l0cytbYiwwLDBdKS9UKS90YXUpOyB0YXVeMiBiZWxvbmdzIGluIHN0dWRlbnQgS0wgbG9zcyIsCiAgInNjb3BlIjogImZpdCBkaWFnbm9zdGljcyBvbmx5OyBubyBpbmRlcGVuZGVudCBwZXJmb3JtYW5jZSBjbGFpbSIsCiAgImZvcmJpZGRlbl9pbnB1dHMiOiBbCiAgICAidmFsX3NlbGVjdCBwcmVkaWN0aW9ucyIsCiAgICAiaGlzdG9yaWNhbF90ZXN0IHByZWRpY3Rpb25zIiwKICAgICJzdXBwbGVtZW50YXJ5XzIwMCBwcmVkaWN0aW9ucyIsCiAgICAiV2lub2dyb3VuZCBzY29yZXMiCiAgXSwKICAiYm91bmRhcnlfcnVsZSI6ICJyZXBvcnQgYm91bmRhcnk7IGRvIG5vdCBleHBhbmQgZ3JpZCBhZnRlciBvYnNlcnZpbmcgZG93bnN0cmVhbSByZXN1bHRzIgp9', 'FILE_SHA256.json': 'ewogICIwMF/lhYjor7tfRGF5Mue7k+aenOS4juS4i+S4gOatpS5tZCI6ICIzYmNmOWU5ODc4NWViMWRhNWMwMjhlOTRkZmZiNWQ3NmU4MzI0MjQ4YTI4ODQwYTdjMTQ4MzMwNmM1MWUwOTE5IiwKICAiRGF5MuivhOS7t+aVsOaNrui3r+e6vy5qc29uIjogImRmNDlhODlhZGZhYzFlZmZiZDYwMmJhZDI5ZTI5YTA1OTc2NDkyODc1MGE3YzYyMjQ1NGRiYzRlMWY5YjhlYzAiLAogICJUZWFjaGVy5qCh5YeGX+aLn+WQiOmbhuiviuaWrS5jc3YiOiAiNDRmNDY4YjhlMzk5MmJmZmYzNDljNDQxMjVmYzY0ZTIwZTgwZWMwMjRmNzQzMDJlNWZjZDE2YWY2NzcwOWQ0NiIsCiAgImJhc2VfZGV2Y2FsX3ByZWRpY3Rpb25zLmpzb25sIjogImRlYzBlNmUzZjJhZTM1NDIzMzc0YmEzNTdkZjk2YzIwMThmZGIyYjgyMDNhY2I4Mzc1YjU2OWZhNDI0NmIwY2YiLAogICJkYXkyX3J1bnRpbWUucHkiOiAiZTY2YjRmYWFhNTRhOGEwZjNiY2VkYjE0OTc0NTQ0OThmYjFkMTg2OWM1YWNjY2U5YjY0NmNlNmZiODk5MTM3ZCIsCiAgImRpYWdub3N0aWNfZGV2X3Jlc2VydmF0aW9uLmpzb24iOiAiZmM1NjUyZTQ5OWJhNjA4OWFlMDc2OWY5YWFhMzA0NDNiYjc3Y2U0Y2YzNjFkYjkwZDMxNjk1Y2RlNmFjZTYxNSIsCiAgImRpYWdub3N0aWNfaW1hZ2Vfc3RhdHVzLmpzb24iOiAiNzM5MDhjOTA4OTI1OWU0NjIyNWJlMDdlNDg4NTdmODYwZGQxYWIzYzhlMDg1N2JhYTU0YjlmMzMyN2ZjYTEwNiIsCiAgImVudmlyb25tZW50Lmpzb24iOiAiYzdiMGQxYTc0OWM2MzU0OGY0OTdkYmFhOGExZjBmY2MzODk4NjQzYWIxNDk1NjQ3ZjU1ZjQ2ZmU1OTgwYjkwMiIsCiAgImlucHV0X2F1ZGl0Lmpzb24iOiAiOTAwMzg3Y2QxNTdmYWZkYzIwNGZmODRjYTRjMWI5YTYxNDFjNzU0OWM3OTMyNGZkOTBlY2Y5MDUyNGI4N2UwYSIsCiAgImlucHV0cy9iYXNlLmpzb25sIjogIjRjNjEwMjM1NWU3MzgyZWY0OTM2ZmU2OWM5OWUwNGVlNjY1MzE5ZTAxYTgxNDM0ZTYyYmZkZWJiYTQwZWNmNTciLAogICJpbnB1dHMvaW1hZ2VfaGFzaF9wcm9qZWN0aW9uLmpzb24iOiAiZDRmMDBhMjhjMGUxNzQwYWE2N2ZmYjY0YTYzNjJhZjVmMTQ3OTU0NTJhYWMwZjIwOTkxNGU5NDg1ZTlhYjhkMCIsCiAgImlucHV0cy9qZXYuanNvbmwiOiAiNmY4NWIyMGYzY2MwY2MzNzAxYmYwZWU2NDliYzdmNmJiYzgyYjU5OGE1MjQyMDlhNWQ3ODQyNDZjZmFiZDQ2YSIsCiAgImlucHV0cy9wcm92ZW5hbmNlLmpzb24iOiAiYTQwOWZjOGMwZjUyN2Q5MDZiNzBiNmExNDdhODIwNWViYzI5MjNmZjAzMTFlNzNhM2RhNmI4OGM3MjlmMGQ4MiIsCiAgImlucHV0cy90ZWFjaGVyX3J1bi5qc29uIjogImE0MjVhMWNjMjgxMWViYmM4NWUxMDMzYmE4NGRhZThhMDYzMmNjZDhlNTNiOGQ3YzVhNGYyZmNlMTkzNTkyOWIiLAogICJpbnB1dHMvdmFsX2ZpdC5qc29ubCI6ICJhNThlMmYwMDA0N2YxNTljYTdiNjQxNTU2NGZmMDNiNjk5YzY4ODk3ZmVhODA1ZmJmYjkwODg3MjlmZGJiYWY0IiwKICAiamV2X2RldmNhbF9wcmVkaWN0aW9ucy5qc29ubCI6ICIxZmZlMDEwOGQxYmNjOTY5OWE5MTdkOGI1ODgyM2UwNzA5ZWNjMGJiYTk2NWM1YjZjZDIzMjlmNTdlZTU0Y2ZlIiwKICAibm90ZWJvb2tfcGF5bG9hZF9wcm92ZW5hbmNlLmpzb24iOiAiNTI3NWJjMjYzYmUzM2VmMjgwOGUzZDU4NGM2MzEyMDk1MmRjYjZlN2IyOGZlNmY3NGJhNGYyOGFmZmU2NmUzZSIsCiAgInByb3RvY29sX0xPQ0tFRC5qc29uIjogImRlOTI2ZjM5ODgyY2MzZTYzNjBhOTEzYTg4N2I0MmY4OTQzODg5MDAwMzE5MzAxMDc5YzE3Yzk3MDE0MWI3YWMiLAogICJ0ZWFjaGVyX2NhbGlicmF0aW9uX3N0YWdlMi5qc29uIjogIjg3NDVkZmUxMTEwMjkxZmY1NGFiYjQ1MzYyYmI5MjJjZjEzY2Q5YzFiYmJkMjQyYmFhODNmZjRiODYyOTlkMjYiLAogICJ0ZWFjaGVyX3RhcmdldF9oZWxwZXIucHkiOiAiOGVjM2YzMmY5MGZkNmEyOWVkYWUzYzM5ZTY0NjU1ZThhZThlYTFkMWZmZGJhZDFkNTY4OGEyMTQ1ZWIzNjQ1NCIsCiAgIuWJqeS9meacquWIhumFjeWbvueJh1/pnZ7mnIDnu4jmtYvor5UuY3N2IjogIjFhNmNkNDBkM2RlODkyNmIzNDllMDc0Y2VjYzZiYTBlN2JmMWRlYzA0MjQ5ZDFhNjgwNTRhMzdhOGQzMmY3ODgiLAogICLlrozmlbTnvZHmoLzmkJzntKLorrDlvZUuY3N2IjogIjQyMjllZTRkZTA0ZDJlZmM2ZTUxMGFiNWFlM2ZhMTNjYzFlY2EyNTQxNGQzZGMwOWNkNTRiNDRmYzNiZTJjNjMiLAogICLni6znq4vpooTmoIfms6hfSGF6ZWwuY3N2IjogIjY0MjczYTk1YzAyOTI4YzE0MjMxNGNjY2JkMjcxYjdmZGM5NTAwYjBhZWZlYzJhMjIxOTAyYTMzYWExYjdiN2UiLAogICLni6znq4vpooTmoIfms6hf6IqZ6IqZ55qE6bG8LmNzdiI6ICJhZmNkZmFiZWZhYjQyNmMxNDljYzViNjI1YzdkOGJjMDRjYjlhODZmY2ZmY2Y3NTdlN2IwMGY1Y2U3MGU3MDZjIiwKICAi6K+K5pat5byA5Y+R5YCZ6YCJX+S4jeaYr+W3suWumueov+mFjeWvuS5jc3YiOiAiODJjZGZhYzE0ODExNmRmNzRiMzQzMTAzMTMyYTJmNDY3YjUwYjVjNGY1ZjcyYmY4OTQzYTFmYjBhZGZmZThmYyIsCiAgIuiviuaWreW8gOWPkeWAmemAiV/nnIvlm77pobUuaHRtbCI6ICJhNmMyNGIxMzZmYWRkODQ3MGRlYjNkMjRjYTJiNGNkNjQ3NWNiOTM5NDBmZTUxNjRiOWIyMjQwZTQyZWYyM2MzIiwKICAi6K+K5pat5qCH5rOo6KeE5YiZX+W3suiOt+iKmeiKmeWOn+WImeiupOWPr1/mlbTkvZPmnKrlhrvnu5MubWQiOiAiYmRlMjcyMGVmNjFmNzAyY2JiOWNjNDkyOWJlMmM1ZGFhMTVhOTM5NWM3MzM3YWQ0MGZiNjE5MjlhYWRmNzY4YiIKfQ=='}
TEACHER_HASHES={'teacher_calibration_stage2.json': '8745dfe1110291ff54abb45362bb922cf13cd9c1bbbd242baa83ff4b86299d26', 'protocol_LOCKED.json': 'de926f39882cc3e6360a913a887b42f8943889000319301079c17c970141b7ac', 'FILE_SHA256.json': '6dfde9f8a716d5b8c596aa956e5ba771185225041956608be2506e9d72245f0d'}
CODE=WORK/'code'; CODE.mkdir(exist_ok=True)
for name,s in SOURCES.items():
    assert hashlib.sha256(s.encode()).hexdigest()==CODE_HASHES[name]
    p=CODE/name
    if p.exists():
        previous=p.read_text(encoding='utf-8')
        if previous!=s:
            permitted=(name=='stage2.py' and hashlib.sha256(previous.encode()).hexdigest()=='b6b7fcbe0664161e50ea56e990b06b6f18709330ac1cf7ab3042ee1bbfc937fe' and not (WORK/'protocol.json').exists())
            assert permitted, '代码发生变化且不属于未启动实验的环境检查修复，请保留原WORK'
            p.write_text(s,encoding='utf-8')
    else: p.write_text(s,encoding='utf-8')
RECEIPT=Path('/content/Stage2_Hazel_teacher_receipt_8745dfe1'); RECEIPT.mkdir(exist_ok=True)
for name,value in TEACHER_BYTES.items():
    b=base64.b64decode(value); assert hashlib.sha256(b).hexdigest()==TEACHER_HASHES[name]
    p=RECEIPT/name
    if p.exists(): assert p.read_bytes()==b, '教师快照变化'
    else: p.write_bytes(b)
sys.path.insert(0,str(CODE))
for name in ['cache_audit','experiment','stage2']:
    spec=importlib.util.spec_from_file_location(name,CODE/(name+'.py'))
    module=importlib.util.module_from_spec(spec);sys.modules[name]=module;spec.loader.exec_module(module)
import stage2,experiment
print('新教师参数与代码已就绪')


## 4．核查 Day2 和数据，锁定本轮18条件

沿用冻结CLIP特征 [v,t,abs(v−t),v*t]，2048→512→3 MLP（ReLU、dropout=.1）。Day2训练预算：30 epochs，batch128，AdamW，weight_decay=1e-4。初始化、每轮批次和dropout随机流按种子配对。

**checkpoint规则**：只在 val_select 按未做学生后校准的 Macro-F1 最大、NLL 最小、较早epoch选最佳模型。val_fit不参与学生checkpoint选择；不加载测试预测。

**损失定义**：

`q_raw = softmax(z_raw / τ)`

`q_cal = softmax(((z_raw + [b,0,0]) / T) / τ)`

`L = (1−α)*CE(student_logits, gold) + α*τ²*KL(q_teacher || softmax(student_logits/τ))`

KL采用batch mean；α是蒸馏项权重。只在教师新校准组使用T/b一次。学生损失内部除τ；指标使用原始学生logits，不继续除τ，不套Day2学生后校准T。Day2既有学生后校准结果保留供后续对照。

新教师交付的拟合集为400条val_fit，训练原始logits版本需匹配。历史校准文件仅因原包完整性审计而读取，**不会成为本轮新KD参数**。


In [ ]:
ctx=stage2.prepare(SOURCE,DAY2,RECEIPT,WORK,legacy=LEGACY)
print('教师新参数：',ctx['params'])
print('公共学习率：',ctx['config']['lr'],'；tau=',ctx['config']['tau'])
display(pd.DataFrame([dict(method=m,alpha=a,seeds=stage2.SEEDS,lr=.0003,tau=2) for a in stage2.ALPHAS for m in stage2.METHODS]))
print('协议已锁定：',WORK/'protocol.json')


## 5．训练18条件，按完整三种子配置推进

顺序：α=.25原始三种子→.25校准三种子→.5原始→.5校准→.75原始→.75校准。每个配置先完成全部三个种子，再启动下一配置。

算力有限时，可设 MAX_NEW_GROUPS=2，先完成同α的两组三种子；下次增加或保持配额继续。配额只限制本次启动的未完成配置，不改变18条件协议，不据阶段分数改搜索范围。MAX_NEW_GROUPS=0只整理成果。

每轮原子保存断点；同一环境、同一WORK重跑可恢复，完成的模型跳过。软中断或GPU内存不足会保留进度、停止启动新配置；意外断连重跑即可。突然终止不能保证报告已写完，恢复后可设0先整理。不要因为单种子分数高低停止配置；不完整三种子不进入比较。

同一WORK只能一个会话写入。ACTIVE.lock只有在确认旧会话已停止后才可手动移除。环境、代码、输入或参数变化会拒绝恢复，应保留旧记录后另建WORK。


In [ ]:
try:
    stage2.run_groups(ctx,max_new_groups=MAX_NEW_GROUPS)
finally:
    REPORT=stage2.report(ctx)
print((REPORT/'status.json').read_text(encoding='utf-8'))
display(pd.read_csv(REPORT/'all_18_conditions.csv'))


## 6．接续Day2统一结果表与三类配对比较

- `combined_Day2_KD_summary.csv`：Day2公共学习率下CE、两种LS及其raw/temperature，与本轮完整三种子KD均值/样本标准差。
- `combined_Day2_KD_per_seed.csv`：全部已完成种子的分数；eligible_for_comparison=false表示该配置缺种子，不用于结论。
- `paired_deltas.csv`：KD−CE、同α校准−原始、同方法较大α−较小α，均按种子一一配对，仅在所需两组均完整时生成。
- `paired_delta_summary.csv`：上述三种子差值的均值/样本标准差。
- `all_18_conditions.csv`、`status.json`：18条件进度及明确缺项；不会因只有部分配置完成而标成全完成。
- `Day2_preserved`：完整Day2监督模型、训练历史、公共学习率选择、LS、学生后校准和原结果表。
- `legacy_KD_lr0001_supplement.csv`：如提供旧结果，单列旧学习率KD；不加入18条件或主要配对检验。

指标：Macro-F1、Accuracy、E/N/C各类F1、NLL、10等宽分箱ECE、Brier（三类平方误差之和）。指标为原值；F1/Accuracy/ECE差值乘100才是百分点，NLL/Brier/ECE越小越好。种子间标准差不是置信区间。val_select已用于选模与历史决策，所有比较是开发性描述，不声称独立测试确认；不自动挑单个最好种子或按测试集改α。


In [ ]:
REPORT=stage2.report(ctx)
display(pd.read_csv(REPORT/'combined_Day2_KD_summary.csv'))
display(pd.read_csv(REPORT/'paired_deltas.csv'))
display(pd.read_csv(REPORT/'paired_delta_summary.csv'))
print('缺项与完成状态：')
print((REPORT/'status.json').read_text(encoding='utf-8'))


## 7．打包本阶段成果

包含运行协议、来源哈希、新教师参数、18条件状态、所有已完成最佳模型/历史/逐条预测、合并Day2表、配对差、Day2原始监督成果及可选旧KD补充。每个模型单独保存best.pt、history.json、done.json、result.json和predictions.csv。

全完成和阶段性完成均可导出，status.json明确区分。ZIP包含PACKAGE_SHA256.json供核对。不包含重复特征、原图和大型优化器state.pt；断点仍留WORK，续跑请使用原WORK，不能仅靠此轻量ZIP恢复未完成训练。老师交付不会重复校准，测试集不会评分。

本 notebook 文件可由你上传到 GitHub 的 Stage2_Hazel 分支；不把Drive训练数据、模型权重或凭据写入代码仓库。


In [ ]:
ZIP=stage2.package(ctx)
print('结果包已保存：',ZIP)
print('正式完成以 report/status.json 的 complete=true、completed_conditions=18 为准。')
